In [1]:
# =========================
# БЛОК 1. Установка библиотек
# =========================

!pip install -q pandas numpy nltk spacy gensim python-Levenshtein openpyxl scikit-learn transformers torch
!python -m spacy download en_core_web_sm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 40.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 48.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# =========================
# БЛОК 1. Импорт библиотек
# =========================

import os
import re
import random
import warnings
from pathlib import Path
from functools import lru_cache
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.corpus import wordnet as wn

import spacy
from Levenshtein import distance as levenshtein_distance

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from google.colab import drive

warnings.filterwarnings("ignore")

nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("stopwords")
nltk.download("punkt")

nlp = spacy.load("en_core_web_sm")
stop_words = set(stopwords.words("english"))

random.seed(42)
np.random.seed(42)

print("Библиотеки загружены.")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Библиотеки загружены.


In [4]:
 # =========================
# БЛОК 1. Подключение Google Drive
# =========================

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
# =========================
# БЛОК 1. Пути к файлам
# =========================

# Укажи здесь путь к папке на Google Drive, где лежат три CSV-файла.
# Пример:
# DATA_DIR = "/content/drive/MyDrive/ВКР/corpora"

DATA_DIR = "/content/drive/MyDrive/вкр"

CORPUS_FILENAME = "final_corpus .csv"
ANSWER_KEY_FILENAME = "answer_key .csv"
REALEC_FILENAME = "ege_lexical_targets_for_realec .csv"

DATA_DIR = Path(DATA_DIR)

CORPUS_PATH = DATA_DIR / CORPUS_FILENAME
ANSWER_KEY_PATH = DATA_DIR / ANSWER_KEY_FILENAME
REALEC_PATH = DATA_DIR / REALEC_FILENAME

for name, path in {
    "final_corpus": CORPUS_PATH,
    "answer_key": ANSWER_KEY_PATH,
    "REALEC": REALEC_PATH
}.items():
    print(name, "->", path, "| exists:", path.exists())

final_corpus -> /content/drive/MyDrive/вкр/final_corpus .csv | exists: True
answer_key -> /content/drive/MyDrive/вкр/answer_key .csv | exists: True
REALEC -> /content/drive/MyDrive/вкр/ege_lexical_targets_for_realec .csv | exists: True


In [6]:
# =========================
# БЛОК 1. Загрузка корпусов
# =========================

if not CORPUS_PATH.exists():
    raise FileNotFoundError(f"Не найден основной корпус: {CORPUS_PATH}")

if not ANSWER_KEY_PATH.exists():
    raise FileNotFoundError(f"Не найден answer_key: {ANSWER_KEY_PATH}")

if not REALEC_PATH.exists():
    raise FileNotFoundError(f"Не найден REALEC-файл: {REALEC_PATH}")

corpus_df = pd.read_csv(CORPUS_PATH)
answer_key_df = pd.read_csv(ANSWER_KEY_PATH)
realec_df = pd.read_csv(REALEC_PATH)

print("corpus_df:", corpus_df.shape)
print("answer_key_df:", answer_key_df.shape)
print("realec_df:", realec_df.shape)

print("\nСтолбцы corpus_df:")
print(corpus_df.columns.tolist())

print("\nСтолбцы answer_key_df:")
print(answer_key_df.columns.tolist())

print("\nСтолбцы realec_df:")
print(realec_df.columns.tolist())

corpus_df: (343, 10)
answer_key_df: (343, 7)
realec_df: (1601, 12)

Столбцы corpus_df:
['kod', 'title', 'task_id', 'distractor_1', 'distractor_2', 'distractor_3', 'target', 'full_text', 'sentence_with_gap', 'sentence_with_answer']

Столбцы answer_key_df:
['kod', 'title', 'task_id', 'distractor_1', 'distractor_2', 'distractor_3', 'target']

Столбцы realec_df:
['target_unit', 'target_head', 'correct_element', 'lexical_category', 'subcategory', 'source_target', 'error_variant', 'error_type', 'learner_sentence', 'correct_sentence', 'comment', 'source_error']


In [7]:
# =========================
# БЛОК 1. Базовая очистка текста
# =========================

def clean_text(text: str) -> str:
    """
    Приводит текст к аккуратному виду:
    - убирает NaN;
    - нормализует кавычки и тире;
    - убирает лишние пробелы.
    """
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.replace("\ufeff", "")
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("—", "-").replace("–", "-")
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def unique_list(items: List[str]) -> List[str]:
    """
    Удаляет повторы, сохраняя исходный порядок.
    """
    result = []
    seen = set()

    for item in items:
        item = clean_text(item).lower()

        if item and item not in seen:
            result.append(item)
            seen.add(item)

    return result


def make_item_key(kod, title, task_id) -> str:
    """
    Создает уникальный идентификатор задания.
    """
    return f"{clean_text(kod)}|{clean_text(title)}|{clean_text(task_id)}"

In [8]:
# =========================
# БЛОК 1. Подготовка рабочего df
# =========================

required_columns = [
    "kod",
    "title",
    "task_id",
    "target",
    "full_text",
    "sentence_with_gap",
    "sentence_with_answer",
    "distractor_1",
    "distractor_2",
    "distractor_3"
]

missing_columns = [
    col for col in required_columns
    if col not in corpus_df.columns
]

if missing_columns:
    raise ValueError(f"В corpus_df не хватает столбцов: {missing_columns}")

df = corpus_df.copy()

df["item_key"] = df.apply(
    lambda row: make_item_key(
        row["kod"],
        row["title"],
        row["task_id"]
    ),
    axis=1
)

df["text_clean"] = df["full_text"].apply(clean_text)
df["sentence_clean"] = df["sentence_with_answer"].apply(clean_text)
df["sentence_with_gap_clean"] = df["sentence_with_gap"].apply(clean_text)
df["target_clean"] = df["target"].apply(clean_text).str.lower()

df["gold_distractors"] = df.apply(
    lambda row: unique_list([
        row["distractor_1"],
        row["distractor_2"],
        row["distractor_3"]
    ]),
    axis=1
)

print("Рабочий df:", df.shape)

df[
    [
        "kod",
        "task_id",
        "target_clean",
        "sentence_clean",
        "gold_distractors"
    ]
].head()

Рабочий df: (343, 16)


,kod,task_id,target_clean,sentence_clean,gold_distractors
0,09B13D,30,announced,"Last week, our coach Mr Brown announced we wer...","[promoted, extended, presented]"
1,09B13D,31,interested,We weren't all that interested in the outdoors.,"[delighted, inspired, excited]"
2,09B13D,32,however,"However, Mr Brown claimed that we'd really enj...","[moreover, although, otherwise]"
3,09B13D,33,thick,"Now we were in a state park, marching through ...","[heavy, strong, rich]"
4,09B13D,34,remind,"Doesn't its shape remind you of an elephant?"" ...","[review, remember, reflect]"


In [9]:
# =========================
# БЛОК 1. Быстрая диагностика корпуса
# =========================

print("Количество заданий:", len(df))
print("Пустые full_text:", df["text_clean"].eq("").sum())
print("Пустые sentence_with_answer:", df["sentence_clean"].eq("").sum())
print("Пустые sentence_with_gap:", df["sentence_with_gap_clean"].eq("").sum())
print("Пустые target:", df["target_clean"].eq("").sum())
print("Дубли item_key:", df["item_key"].duplicated().sum())

df["target_found_in_sentence"] = df.apply(
    lambda row: bool(
        re.search(
            r"\b" + re.escape(row["target_clean"]) + r"\b",
            row["sentence_clean"],
            flags=re.IGNORECASE
        )
    ),
    axis=1
)

print("Target не найден в sentence_with_answer:", (~df["target_found_in_sentence"]).sum())

df.loc[
    ~df["target_found_in_sentence"],
    ["kod", "task_id", "target_clean", "sentence_clean"]
].head(10)

Количество заданий: 343
Пустые full_text: 0
Пустые sentence_with_answer: 0
Пустые sentence_with_gap: 0
Пустые target: 0
Дубли item_key: 0
Target не найден в sentence_with_answer: 20


,kod,task_id,target_clean,sentence_clean
3,09B13D,33,thick,"Now we were in a state park, marching through ..."
14,15e618,30,recognised,Daphne recognisedthem all and stood rapt until...
16,15e618,32,tour,"After breakfast, she walked up to the castle, ..."
17,15e618,33,trouble,Changing times had continued to diminish their...
19,15e618,35,afford,The family couldn'tafford it.
38,207FE9,35,enjoy,"I unwrapped a corner of it and took a bite, pr..."
57,28EF4E,31,tell,She didn't telleither of her brothers who she ...
58,28EF4E,32,hardly,"Emily could hardlywait until her day off, and ..."
59,28EF4E,33,occupied,She had chosen her time carefully: late mornin...
60,28EF4E,34,remember,He pointed indifferently towards the red-brick...


In [10]:
# =========================
# БЛОК 1.1. Исправление склеек вокруг target
# =========================

def repair_gap_spacing(sentence_with_gap: str) -> str:
    """
    Исправляет склейки вокруг пропуска:
    33 ______as -> 33 ______ as
    word30 ______ -> word 30 ______
    """
    s = clean_text(sentence_with_gap)

    if not s:
        return ""

    s = re.sub(r"(\d{1,2}\s*_{2,})(?=[A-Za-z])", r"\1 ", s)
    s = re.sub(r"(?<=[A-Za-z])(\d{1,2}\s*_{2,})", r" \1", s)

    return clean_text(s)


def ensure_target_separated(sentence: str, target: str) -> str:
    """
    Если target склеен с соседним словом, пробуем отделить его пробелами.

    Пример:
    thickas -> thick as
    announcedwe -> announced we

    Важно: функция применяется только если target НЕ найден как отдельное слово.
    """
    s = clean_text(sentence)
    t = clean_text(target).lower()

    if not s or not t:
        return s

    exact_pattern = r"\b" + re.escape(t) + r"\b"

    if re.search(exact_pattern, s, flags=re.IGNORECASE):
        return s

    # target склеен с буквами слева и справа
    s = re.sub(
        rf"(?i)(?<=\w)({re.escape(t)})(?=\w)",
        r" \1 ",
        s
    )

    # target склеен с буквами слева
    s = re.sub(
        rf"(?i)(?<=\w)({re.escape(t)})\b",
        r" \1",
        s
    )

    # target склеен с буквами справа
    s = re.sub(
        rf"(?i)\b({re.escape(t)})(?=\w)",
        r"\1 ",
        s
    )

    return clean_text(s)


df["sentence_clean"] = df.apply(
    lambda row: ensure_target_separated(
        row["sentence_clean"],
        row["target_clean"]
    ),
    axis=1
)

df["sentence_with_gap_clean"] = df["sentence_with_gap_clean"].apply(
    repair_gap_spacing
)

df["target_found_in_sentence"] = df.apply(
    lambda row: bool(
        re.search(
            r"\b" + re.escape(row["target_clean"]) + r"\b",
            row["sentence_clean"],
            flags=re.IGNORECASE
        )
    ),
    axis=1
)

print("Target не найден после исправления:", (~df["target_found_in_sentence"]).sum())

df.loc[
    ~df["target_found_in_sentence"],
    ["kod", "task_id", "target_clean", "sentence_clean"]
].head(20)

Target не найден после исправления: 0


,kod,task_id,target_clean,sentence_clean


In [11]:
# =========================
# БЛОК 2. Загрузка GloVe
# =========================

import gensim.downloader as api

GLOVE_MODEL_NAME = "glove-wiki-gigaword-100"

glove_model = api.load(GLOVE_MODEL_NAME)

print("GloVe загружена:", GLOVE_MODEL_NAME)
print("Размер словаря:", len(glove_model.key_to_index))

[==================================================] 100.0% 128.1/128.1MB downloaded
GloVe загружена: glove-wiki-gigaword-100
Размер словаря: 400000


In [12]:
# =========================
# БЛОК 2. Векторная близость
# =========================

def get_vector_for_expression(expression: str, model):
    """
    Получает вектор для слова или словосочетания.

    Если expression состоит из нескольких слов,
    берется средний вектор известных слов.
    """
    expression = clean_text(expression).lower()
    parts = expression.split()

    vectors = []

    for part in parts:
        if part in model:
            vectors.append(model[part])

    if not vectors:
        return None

    return np.mean(vectors, axis=0)


def vector_similarity(word1: str, word2: str, model) -> float:
    """
    Считает cosine similarity между target и candidate.
    """
    vec1 = get_vector_for_expression(word1, model)
    vec2 = get_vector_for_expression(word2, model)

    if vec1 is None or vec2 is None:
        return 0.0

    similarity = np.dot(vec1, vec2) / (
        np.linalg.norm(vec1) * np.linalg.norm(vec2)
    )

    return float(similarity)

In [13]:
# =========================
# БЛОК 2. Получение 50–100 семантических кандидатов
# =========================

SEMANTIC_TOPN = 100


def get_semantic_candidates(
    target: str,
    model,
    topn: int = 100
) -> List[Tuple[str, float]]:
    """
    Возвращает topn ближайших кандидатов к target по GloVe.

    На этом этапе мы НЕ решаем, хороший это дистрактор или нет.
    Мы просто собираем широкий список семантически близких слов.
    """
    target = clean_text(target).lower()

    if target not in model:
        return []

    candidates = []

    for candidate, similarity in model.most_similar(target, topn=topn):
        candidate = clean_text(candidate).lower()

        # Убираем технический мусор
        if not candidate:
            continue

        if "'" in candidate:
            continue

        if not candidate.replace("-", "").isalpha():
            continue

        candidates.append(
            (candidate, float(similarity))
        )

    return candidates

In [15]:
# =========================
# БЛОК 2. Применяем к корпусу
# =========================

df["semantic_candidates_with_scores"] = df["target_clean"].apply(
    lambda target: get_semantic_candidates(
        target,
        glove_model,
        topn=SEMANTIC_TOPN
    )
)

df["semantic_candidates"] = df["semantic_candidates_with_scores"].apply(
    lambda pairs: [candidate for candidate, score in pairs]
)

df[
    [
        "target_clean",
        "semantic_candidates"
    ]
].head(10)

,target_clean,semantic_candidates
0,announced,"[announcing, plans, announcement, announce, ag..."
1,interested,"[concerned, aware, talking, besides, looking, ..."
2,however,"[although, though, but, because, nevertheless,..."
3,thick,"[dense, thin, covered, tall, layer, dark, laye..."
4,remind,"[reminding, reminded, tell, reminds, assure, a..."
5,observing,"[observe, observed, monitoring, conducting, ob..."
6,told,"[reporters, asked, said, interview, saying, af..."
7,formed,"[forming, merged, established, joined, formati..."
8,introduced,"[adopted, introduction, originally, developed,..."
9,attracted,"[attracting, attracts, attract, enthusiastic, ..."


In [ ]:
# =========================
# БЛОК 2. Диагностика количества кандидатов
# =========================

df["num_semantic_candidates"] = df["semantic_candidates"].apply(len)

print(df["num_semantic_candidates"].describe())

df[
    [
        "target_clean",
        "num_semantic_candidates",
        "semantic_candidates"
    ]
].head(20)

count    343.000000
mean      95.737609
std       12.628334
min        0.000000
25%       96.000000
50%       98.000000
75%      100.000000
max      100.000000
Name: num_semantic_candidates, dtype: float64


,target_clean,num_semantic_candidates,semantic_candidates
0,announced,81,"[announcing, plans, announcement, announce, ag..."
1,interested,98,"[concerned, aware, talking, besides, looking, ..."
2,however,98,"[although, though, but, because, nevertheless,..."
3,thick,98,"[dense, thin, covered, tall, layer, dark, laye..."
4,remind,99,"[reminding, reminded, tell, reminds, assure, a..."
5,observing,100,"[observe, observed, monitoring, conducting, ob..."
6,told,98,"[reporters, asked, said, interview, saying, af..."
7,formed,96,"[forming, merged, established, joined, formati..."
8,introduced,86,"[adopted, introduction, originally, developed,..."
9,attracted,100,"[attracting, attracts, attract, enthusiastic, ..."


In [ ]:
df[["target_clean", "num_semantic_candidates", "semantic_candidates"]].head(10)

,target_clean,num_semantic_candidates,semantic_candidates
0,announced,81,"[announcing, plans, announcement, announce, ag..."
1,interested,98,"[concerned, aware, talking, besides, looking, ..."
2,however,98,"[although, though, but, because, nevertheless,..."
3,thick,98,"[dense, thin, covered, tall, layer, dark, laye..."
4,remind,99,"[reminding, reminded, tell, reminds, assure, a..."
5,observing,100,"[observe, observed, monitoring, conducting, ob..."
6,told,98,"[reporters, asked, said, interview, saying, af..."
7,formed,96,"[forming, merged, established, joined, formati..."
8,introduced,86,"[adopted, introduction, originally, developed,..."
9,attracted,100,"[attracting, attracts, attract, enthusiastic, ..."


In [16]:
# =========================
# БЛОК 3. Анализ target в контексте
# =========================

def analyze_target_in_context(sentence: str, target: str) -> Dict[str, str]:
    """
    Находит target в предложении и определяет:
    - lemma: начальную форму;
    - pos: крупную часть речи, например VERB / ADJ / NOUN;
    - tag: точный грамматический тег, например VBD / VBN / JJ;
    - dep: синтаксическую роль.
    """
    sentence = clean_text(sentence)
    target = clean_text(target).lower()

    doc = nlp(sentence)

    for i, token in enumerate(doc):
        if token.text.lower() == target:
            return {
                "text": token.text.lower(),
                "lemma": token.lemma_.lower(),
                "pos": token.pos_,
                "tag": token.tag_,
                "dep": token.dep_
            }

    # Если вдруг не нашли target в предложении, анализируем само слово.
    target_doc = nlp(target)

    if len(target_doc) == 0:
        return {
            "text": target,
            "lemma": target,
            "pos": "UNKNOWN",
            "tag": "UNKNOWN",
            "dep": "UNKNOWN"
        }

    token = target_doc[0]

    return {
        "text": token.text.lower(),
        "lemma": token.lemma_.lower(),
        "pos": token.pos_,
        "tag": token.tag_,
        "dep": token.dep_
    }


df["target_analysis"] = df.apply(
    lambda row: analyze_target_in_context(
        row["sentence_clean"],
        row["target_clean"]
    ),
    axis=1
)

df["target_lemma"] = df["target_analysis"].apply(lambda x: x["lemma"])
df["target_pos"] = df["target_analysis"].apply(lambda x: x["pos"])
df["target_tag"] = df["target_analysis"].apply(lambda x: x["tag"])
df["target_dep"] = df["target_analysis"].apply(lambda x: x["dep"])

df[
    [
        "target_clean",
        "target_lemma",
        "target_pos",
        "target_tag",
        "target_dep"
    ]
].head(20)

,target_clean,target_lemma,target_pos,target_tag,target_dep
0,announced,announce,VERB,VBD,ROOT
1,interested,interested,ADJ,JJ,acomp
2,however,however,ADV,RB,advmod
3,thick,thick,ADJ,JJ,advmod
4,remind,remind,VERB,VB,ROOT
5,observing,observe,VERB,VBG,amod
6,told,tell,VERB,VBD,ccomp
7,formed,form,VERB,VBD,ROOT
8,introduced,introduce,VERB,VBD,relcl
9,attracted,attract,VERB,VBD,conj


In [17]:
# =========================
# БЛОК 3. Анализ кандидата
# =========================

@lru_cache(maxsize=50000)
def cached_word_analysis(word: str) -> Tuple[str, str, str]:
    """
    Быстрый анализ отдельного слова:
    возвращает lemma, pos, tag.
    """
    word = clean_text(word).lower()
    doc = nlp(word)

    if len(doc) == 0:
        return word, "UNKNOWN", "UNKNOWN"

    token = doc[0]

    return token.lemma_.lower(), token.pos_, token.tag_


def get_candidate_lemma(candidate: str) -> str:
    return cached_word_analysis(candidate)[0]


def get_candidate_pos(candidate: str) -> str:
    return cached_word_analysis(candidate)[1]


def get_candidate_tag(candidate: str) -> str:
    return cached_word_analysis(candidate)[2]

In [22]:
# =========================
# БЛОК 3. Грамматические группы
# =========================

DISCOURSE_MARKERS = {
    "however", "although", "though", "therefore", "moreover",
    "nevertheless", "otherwise", "instead", "meanwhile", "besides"
}

PREPOSITIONS_AND_PARTICLES = {
    "in", "on", "at", "by", "for", "from", "with", "into", "onto",
    "out", "up", "off", "away", "back", "down", "over", "around"
}


def get_form_group(word: str, pos: str, tag: str) -> str:
    """
    Переводит pos/tag в более полезную для ЕГЭ грамматическую группу.
    """
    word = clean_text(word).lower()

    if word in DISCOURSE_MARKERS:
        return "discourse_marker"

    if word in PREPOSITIONS_AND_PARTICLES:
        return "preposition_or_particle"

    # announced, told, formed
    if tag == "VBD":
        return "verb_past"

    # interested, disappointed, introduced
    if tag == "VBN":
        return "participle_past"

    # observing, looking, thinking
    if tag == "VBG":
        return "participle_ing"

    # happy, thick, important
    if tag in {"JJ", "JJR", "JJS"}:
        # Если слово выглядит как причастие, лучше считать его participial adjective.
        if word.endswith("ed") or word.endswith("en"):
            return "participle_past"
        if word.endswith("ing"):
            return "participle_ing"
        return "adjective"

    # however часто может быть RB
    if tag in {"RB", "RBR", "RBS"}:
        if word in DISCOURSE_MARKERS:
            return "discourse_marker"
        return "adverb"

    if pos == "NOUN":
        return "noun"

    if pos == "VERB":
        return "verb"

    return pos.lower()

In [23]:
# =========================
# БЛОК 3. Группа target
# =========================

df["target_form_group"] = df.apply(
    lambda row: get_form_group(
        row["target_clean"],
        row["target_pos"],
        row["target_tag"]
    ),
    axis=1
)

df[
    [
        "target_clean",
        "target_pos",
        "target_tag",
        "target_form_group"
    ]
].head(20)

,target_clean,target_pos,target_tag,target_form_group
0,announced,VERB,VBD,verb_past
1,interested,ADJ,JJ,participle_past
2,however,ADV,RB,discourse_marker
3,thick,ADJ,JJ,adjective
4,remind,VERB,VB,verb
5,observing,VERB,VBG,participle_ing
6,told,VERB,VBD,verb_past
7,formed,VERB,VBD,verb_past
8,introduced,VERB,VBD,verb_past
9,attracted,VERB,VBD,verb_past


In [24]:
# =========================
# БЛОК 3. Грамматическая совместимость candidate и target
# =========================

def is_same_lemma(candidate: str, target: str) -> bool:
    """
    Проверяет, не является ли кандидат формой того же слова.
    Например:
    announce / announced / announcing
    remind / reminded / reminding
    """
    candidate = clean_text(candidate).lower()
    target = clean_text(target).lower()

    return get_candidate_lemma(candidate) == get_candidate_lemma(target)


def candidate_form_group(candidate: str) -> str:
    """
    Определяет грамматическую группу кандидата.
    """
    candidate = clean_text(candidate).lower()

    lemma, pos, tag = cached_word_analysis(candidate)

    return get_form_group(candidate, pos, tag)


def is_grammar_compatible(candidate: str, row) -> bool:
    """
    Проверяет, подходит ли candidate по грамматической форме к target.
    """
    candidate = clean_text(candidate).lower()
    target = row["target_clean"]

    if not candidate or candidate == target:
        return False

    if is_same_lemma(candidate, target):
        return False

    target_group = row["target_form_group"]
    candidate_group = candidate_form_group(candidate)

    # however -> therefore / moreover / otherwise
    if target_group == "discourse_marker":
        return candidate_group == "discourse_marker"

    # out -> off / up / away
    if target_group == "preposition_or_particle":
        return candidate_group == "preposition_or_particle"

    # interested -> excited / delighted / inspired
    if target_group == "participle_past":
        return candidate_group == "participle_past"

    # observing -> monitoring / conducting / analyzing
    if target_group == "participle_ing":
        return candidate_group == "participle_ing"

    # announced / told / formed
    if target_group == "verb_past":
        return candidate_group in {"verb_past", "participle_past"}

    # thick / happy / important
    if target_group == "adjective":
        return candidate_group == "adjective"

    # reason / tour / trouble
    if target_group == "noun":
        return candidate_group == "noun"

    # remind / imagine / afford
    if target_group == "verb":
        return candidate_group == "verb"

    # definitely
    if target_group == "adverb":
        return candidate_group == "adverb"

    return candidate_group == target_group

In [27]:
df["num_semantic_candidates"] = df["semantic_candidates"].apply(len)

In [28]:
# =========================
# БЛОК 3. Применяем грамматический фильтр
# =========================

df["semantic_candidates_grammar"] = df.apply(
    lambda row: [
        candidate
        for candidate in row["semantic_candidates"]
        if is_grammar_compatible(candidate, row)
    ],
    axis=1
)

df["num_semantic_candidates_grammar"] = df["semantic_candidates_grammar"].apply(len)

df[
    [
        "target_clean",
        "target_form_group",
        "num_semantic_candidates",
        "num_semantic_candidates_grammar",
        "semantic_candidates_grammar"
    ]
].head(20)

,target_clean,target_form_group,num_semantic_candidates,num_semantic_candidates_grammar,semantic_candidates_grammar
0,announced,verb_past,81,20,"[agreed, planned, decided, expected, unveiled,..."
1,interested,participle_past,98,3,"[involved, keen, realized]"
2,however,discourse_marker,98,6,"[although, though, nevertheless, therefore, mo..."
3,thick,adjective,98,25,"[dense, thin, tall, dark, yellow, dry, shiny, ..."
4,remind,verb,99,42,"[tell, ask, forget, wish, inform, remember, im..."
5,observing,participle_ing,100,58,"[monitoring, conducting, analyzing, photograph..."
6,told,verb_past,98,33,"[asked, said, quoted, insisted, spoke, informe..."
7,formed,verb_past,96,30,"[merged, established, joined, created, founded..."
8,introduced,verb_past,86,29,"[adopted, developed, created, modified, used, ..."
9,attracted,verb_past,100,19,"[drew, garnered, encouraged, generated, grew, ..."


In [29]:
# =========================
# БЛОК 3. Диагностика после грамматического фильтра
# =========================

for _, row in df.head(12).iterrows():
    print("=" * 100)
    print("target:", row["target_clean"])
    print("target_group:", row["target_form_group"])
    print("до фильтра:", row["semantic_candidates"][:15])
    print("после фильтра:", row["semantic_candidates_grammar"][:15])

target: announced
target_group: verb_past
до фильтра: ['announcing', 'plans', 'announcement', 'announce', 'agreed', 'recently', 'tuesday', 'thursday', 'wednesday', 'monday', 'month', 'earlier', 'planned', 'decided', 'week']
после фильтра: ['agreed', 'planned', 'decided', 'expected', 'unveiled', 'proposed', 'confirmed', 'approved', 'promised', 'scheduled', 'signed', 'rejected', 'issued', 'launched', 'offered']
target: interested
target_group: participle_past
до фильтра: ['concerned', 'aware', 'talking', 'besides', 'looking', 'thinking', 'focused', 'doing', 'discussing', 'wanting', 'studying', 'involved', 'idea', 'consider', 'considering']
после фильтра: ['involved', 'keen', 'realized']
target: however
target_group: discourse_marker
до фильтра: ['although', 'though', 'but', 'because', 'nevertheless', 'only', 'result', 'that', 'not', 'this', 'yet', 'fact', 'later', 'as', 'both']
после фильтра: ['although', 'though', 'nevertheless', 'therefore', 'moreover', 'instead']
target: thick
target_

In [30]:
# =========================
# БЛОК 4. Проверка появления кандидата в тексте
# =========================

def appears_as_word(candidate: str, text: str) -> bool:
    """
    Проверяет, встречается ли candidate в тексте как отдельное слово.

    Например:
    candidate = "talk"
    text = "talking" -> False
    text = "talk to me" -> True
    """
    candidate = clean_text(candidate).lower()
    text = clean_text(text).lower()

    if not candidate or not text:
        return False

    pattern = r"\b" + re.escape(candidate) + r"\b"

    return bool(
        re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )
    )


def is_allowed_stopword_candidate(candidate: str, row) -> bool:
    """
    Обычно стоп-слова плохие кандидаты.

    Но если target сам является предлогом / частицей,
    то кандидаты тоже могут быть служебными словами:
    out -> off / up / away
    at -> in / on / by
    """
    candidate = clean_text(candidate).lower()
    target_group = row["target_form_group"]

    if candidate not in stop_words:
        return True

    if target_group in {
        "preposition_or_particle",
        "discourse_marker"
    }:
        return True

    return False

In [31]:
# =========================
# БЛОК 4. Причина удаления по контексту
# =========================

def context_filter_reason(candidate: str, row) -> Optional[str]:
    """
    Возвращает причину, почему кандидат нужно удалить.
    Если кандидат можно оставить, возвращает None.
    """
    candidate = clean_text(candidate).lower()

    if not candidate:
        return "empty_candidate"

    if not is_allowed_stopword_candidate(candidate, row):
        return "stopword"

    if appears_as_word(candidate, row["sentence_clean"]):
        return "candidate_already_in_sentence"

    if appears_as_word(candidate, row["text_clean"]):
        return "candidate_already_in_full_text"

    return None


def apply_context_filter(row) -> List[str]:
    """
    Оставляет только кандидатов, которые не встречаются
    в предложении и полном тексте.
    """
    result = []

    for candidate in row["semantic_candidates_grammar"]:
        reason = context_filter_reason(candidate, row)

        if reason is None:
            result.append(candidate)

    return unique_list(result)


def collect_removed_by_context(row) -> List[Dict[str, str]]:
    """
    Сохраняет удаленные кандидаты и причину удаления.
    Это полезно для диагностики и описания в ВКР.
    """
    removed = []

    for candidate in row["semantic_candidates_grammar"]:
        reason = context_filter_reason(candidate, row)

        if reason is not None:
            removed.append(
                {
                    "candidate": candidate,
                    "reason": reason
                }
            )

    return removed

In [33]:
# =========================
# БЛОК 4. Применяем контекстный фильтр
# =========================

df["semantic_candidates_context"] = df.apply(
    apply_context_filter,
    axis=1
)

df["removed_by_context_filter"] = df.apply(
    collect_removed_by_context,
    axis=1
)

df["num_semantic_candidates_context"] = df["semantic_candidates_context"].apply(len)

df[
    [
        "target_clean",
        "target_form_group",
        "num_semantic_candidates_grammar",
        "num_semantic_candidates_context",
        "semantic_candidates_context"
    ]
].head(20)

,target_clean,target_form_group,num_semantic_candidates_grammar,num_semantic_candidates_context,semantic_candidates_context
0,announced,verb_past,20,20,"[agreed, planned, decided, expected, unveiled,..."
1,interested,participle_past,3,3,"[involved, keen, realized]"
2,however,discourse_marker,6,6,"[although, though, nevertheless, therefore, mo..."
3,thick,adjective,25,25,"[dense, thin, tall, dark, yellow, dry, shiny, ..."
4,remind,verb,42,41,"[tell, ask, forget, wish, inform, remember, im..."
5,observing,participle_ing,58,57,"[monitoring, conducting, analyzing, photograph..."
6,told,verb_past,33,30,"[asked, said, quoted, insisted, spoke, informe..."
7,formed,verb_past,30,27,"[merged, established, joined, created, founded..."
8,introduced,verb_past,29,27,"[adopted, created, modified, used, changed, pr..."
9,attracted,verb_past,19,17,"[drew, garnered, encouraged, generated, grew, ..."


In [35]:
# =========================
# БЛОК 4. Диагностика после контекстной фильтрации
# =========================

for _, row in df.head(20).iterrows():
    print("=" * 100)
    print("target:", row["target_clean"])
    print("target_group:", row["target_form_group"])
    print("после грамматики:", row["semantic_candidates_grammar"][:15])
    print("после контекста:", row["semantic_candidates_context"][:15])
    print("удалено:", row["removed_by_context_filter"][:8])

target: announced
target_group: verb_past
после грамматики: ['agreed', 'planned', 'decided', 'expected', 'unveiled', 'proposed', 'confirmed', 'approved', 'promised', 'scheduled', 'signed', 'rejected', 'issued', 'launched', 'offered']
после контекста: ['agreed', 'planned', 'decided', 'expected', 'unveiled', 'proposed', 'confirmed', 'approved', 'promised', 'scheduled', 'signed', 'rejected', 'issued', 'launched', 'offered']
удалено: []
target: interested
target_group: participle_past
после грамматики: ['involved', 'keen', 'realized']
после контекста: ['involved', 'keen', 'realized']
удалено: []
target: however
target_group: discourse_marker
после грамматики: ['although', 'though', 'nevertheless', 'therefore', 'moreover', 'instead']
после контекста: ['although', 'though', 'nevertheless', 'therefore', 'moreover', 'instead']
удалено: []
target: thick
target_group: adjective
после грамматики: ['dense', 'thin', 'tall', 'dark', 'yellow', 'dry', 'shiny', 'deep', 'rough', 'gray', 'large', 'sturdy

In [36]:
# =========================
# БЛОК 5. WordNet: синонимы и антонимы
# =========================

def spacy_pos_to_wordnet(pos: str):
    """
    Переводит spaCy POS в WordNet POS.
    """
    mapping = {
        "NOUN": wn.NOUN,
        "VERB": wn.VERB,
        "ADJ": wn.ADJ,
        "ADV": wn.ADV
    }

    return mapping.get(pos)


def get_wordnet_synonyms(word: str, pos: str) -> set:
    """
    Возвращает прямые синонимы слова из WordNet.
    """
    word = clean_text(word).lower()
    wn_pos = spacy_pos_to_wordnet(pos)

    if wn_pos is None:
        return set()

    synonyms = set()

    for synset in wn.synsets(word, pos=wn_pos):
        for lemma in synset.lemmas():
            synonyms.add(
                lemma.name()
                .replace("_", " ")
                .lower()
            )

    synonyms.discard(word)

    return synonyms


def share_wordnet_synset(word1: str, word2: str, pos: str) -> bool:
    """
    Проверяет, находятся ли два слова в одном WordNet-синсете.
    Это сильный признак почти-эквивалентности.
    """
    wn_pos = spacy_pos_to_wordnet(pos)

    if wn_pos is None:
        return False

    synsets1 = set(
        wn.synsets(
            clean_text(word1).lower(),
            pos=wn_pos
        )
    )

    synsets2 = set(
        wn.synsets(
            clean_text(word2).lower(),
            pos=wn_pos
        )
    )

    return bool(synsets1 & synsets2)


def get_wordnet_antonyms(word: str, pos: str) -> set:
    """
    Возвращает антонимы слова из WordNet.
    """
    word = clean_text(word).lower()
    wn_pos = spacy_pos_to_wordnet(pos)

    if wn_pos is None:
        return set()

    antonyms = set()

    for synset in wn.synsets(word, pos=wn_pos):
        for lemma in synset.lemmas():
            for antonym in lemma.antonyms():
                antonyms.add(
                    antonym.name()
                    .replace("_", " ")
                    .lower()
                )

    return antonyms


def is_antonym_candidate(candidate: str, target: str, target_pos: str) -> bool:
    """
    Проверяет, является ли candidate антонимом target.
    """
    candidate = clean_text(candidate).lower()
    target = clean_text(target).lower()

    return (
        candidate in get_wordnet_antonyms(target, target_pos)
        or target in get_wordnet_antonyms(candidate, target_pos)
    )

In [37]:
# =========================
# БЛОК 5. Ручные группы полярности
# =========================

POSITIVE_EMOTION_WORDS = {
    "happy", "glad", "pleased", "delighted", "excited",
    "satisfied", "confident", "relaxed", "proud", "cheerful",
    "joyful", "thrilled", "content", "grateful", "hopeful",
    "calm", "interested", "inspired", "enthusiastic"
}

NEGATIVE_EMOTION_WORDS = {
    "sad", "angry", "tired", "disappointed", "frustrated",
    "confused", "upset", "worried", "afraid", "anxious",
    "nervous", "shocked", "depressed", "annoyed", "bored",
    "guilty", "unhappy"
}


def get_emotion_polarity(word: str) -> Optional[str]:
    """
    Возвращает positive / negative для эмоциональной лексики.
    """
    word = clean_text(word).lower()

    if word in POSITIVE_EMOTION_WORDS:
        return "positive"

    if word in NEGATIVE_EMOTION_WORDS:
        return "negative"

    return None


def has_opposite_emotion_polarity(candidate: str, target: str) -> bool:
    """
    Убирает пары типа:
    disappointed -> delighted
    happy -> sad

    Они часто слишком очевидные и не подходят как качественные дистракторы.
    """
    candidate_polarity = get_emotion_polarity(candidate)
    target_polarity = get_emotion_polarity(target)

    return (
        candidate_polarity is not None
        and target_polarity is not None
        and candidate_polarity != target_polarity
    )

In [38]:
# =========================
# БЛОК 5. Фильтр почти-эквивалентов
# =========================

MAX_ALLOWED_VECTOR_SIMILARITY = 0.82


def near_equivalent_reason(candidate: str, row) -> Optional[str]:
    """
    Возвращает причину удаления кандидата,
    если он слишком близок к target.
    """
    candidate = clean_text(candidate).lower()
    target = row["target_clean"]
    target_pos = row["target_pos"]

    similarity = vector_similarity(
        target,
        candidate,
        glove_model
    )

    # Прямой синоним WordNet
    if candidate in get_wordnet_synonyms(target, target_pos):
        return "wordnet_synonym"

    # То же значение в WordNet
    if share_wordnet_synset(candidate, target, target_pos):
        return "same_wordnet_synset"

    # Антоним
    if is_antonym_candidate(candidate, target, target_pos):
        return "wordnet_antonym"

    # Противоположная эмоциональная полярность
    if has_opposite_emotion_polarity(candidate, target):
        return "opposite_emotion_polarity"

    # Слишком высокая векторная близость
    if similarity > MAX_ALLOWED_VECTOR_SIMILARITY:
        return "too_high_vector_similarity"

    return None


def apply_near_equivalent_filter(row) -> List[str]:
    """
    Убирает кандидаты, которые слишком близки к target
    или являются очевидными антонимами.
    """
    result = []

    for candidate in row["semantic_candidates_context"]:
        reason = near_equivalent_reason(candidate, row)

        if reason is None:
            result.append(candidate)

    return unique_list(result)


def collect_removed_by_near_equivalent_filter(row) -> List[Dict[str, str]]:
    """
    Сохраняет удаленные кандидаты и причину.
    """
    removed = []

    for candidate in row["semantic_candidates_context"]:
        reason = near_equivalent_reason(candidate, row)

        if reason is not None:
            removed.append(
                {
                    "candidate": candidate,
                    "reason": reason,
                    "similarity": round(
                        vector_similarity(
                            row["target_clean"],
                            candidate,
                            glove_model
                        ),
                        4
                    )
                }
            )

    return removed

In [39]:
# =========================
# БЛОК 5. Применяем фильтр почти-эквивалентов
# =========================

df["semantic_candidates_clean"] = df.apply(
    apply_near_equivalent_filter,
    axis=1
)

df["removed_by_near_equivalent_filter"] = df.apply(
    collect_removed_by_near_equivalent_filter,
    axis=1
)

df["num_semantic_candidates_clean"] = df["semantic_candidates_clean"].apply(len)

df[
    [
        "target_clean",
        "num_semantic_candidates_context",
        "num_semantic_candidates_clean",
        "semantic_candidates_clean"
    ]
].head(20)

,target_clean,num_semantic_candidates_context,num_semantic_candidates_clean,semantic_candidates_clean
0,announced,20,20,"[agreed, planned, decided, expected, unveiled,..."
1,interested,3,3,"[involved, keen, realized]"
2,however,6,3,"[therefore, moreover, instead]"
3,thick,25,22,"[tall, dark, yellow, dry, shiny, rough, gray, ..."
4,remind,41,41,"[tell, ask, forget, wish, inform, remember, im..."
5,observing,57,52,"[monitoring, conducting, analyzing, photograph..."
6,told,30,27,"[quoted, insisted, spoke, informed, explained,..."
7,formed,27,26,"[merged, established, joined, created, founded..."
8,introduced,27,27,"[adopted, created, modified, used, changed, pr..."
9,attracted,17,15,"[garnered, encouraged, generated, grew, grown,..."


In [40]:
# =========================
# БЛОК 5. Диагностика
# =========================

for _, row in df.head(12).iterrows():
    print("=" * 100)
    print("target:", row["target_clean"])
    print("после контекста:", row["semantic_candidates_context"][:15])
    print("после фильтра эквивалентов:", row["semantic_candidates_clean"][:15])
    print("удалено:", row["removed_by_near_equivalent_filter"][:8])

target: announced
после контекста: ['agreed', 'planned', 'decided', 'expected', 'unveiled', 'proposed', 'confirmed', 'approved', 'promised', 'scheduled', 'signed', 'rejected', 'issued', 'launched', 'offered']
после фильтра эквивалентов: ['agreed', 'planned', 'decided', 'expected', 'unveiled', 'proposed', 'confirmed', 'approved', 'promised', 'scheduled', 'signed', 'rejected', 'issued', 'launched', 'offered']
удалено: []
target: interested
после контекста: ['involved', 'keen', 'realized']
после фильтра эквивалентов: ['involved', 'keen', 'realized']
удалено: []
target: however
после контекста: ['although', 'though', 'nevertheless', 'therefore', 'moreover', 'instead']
после фильтра эквивалентов: ['therefore', 'moreover', 'instead']
удалено: [{'candidate': 'although', 'reason': 'too_high_vector_similarity', 'similarity': 0.9658}, {'candidate': 'though', 'reason': 'too_high_vector_similarity', 'similarity': 0.9302}, {'candidate': 'nevertheless', 'reason': 'wordnet_synonym', 'similarity': 0.8

In [41]:
# =========================
# БЛОК 5.1. Исправляем фильтр для служебных слов
# =========================

def near_equivalent_reason(candidate: str, row) -> Optional[str]:
    """
    Возвращает причину удаления кандидата,
    если он слишком близок к target.

    Важно:
    для discourse markers, предлогов и частиц НЕ используем
    фильтр высокой векторной близости, потому что для них
    близость как раз ожидаема:
    however -> therefore / moreover
    out -> up / off / away
    """
    candidate = clean_text(candidate).lower()
    target = row["target_clean"]
    target_pos = row["target_pos"]
    target_group = row["target_form_group"]

    similarity = vector_similarity(
        target,
        candidate,
        glove_model
    )

    # Для служебных слов не применяем high similarity как причину удаления.
    skip_high_similarity_filter = target_group in {
        "discourse_marker",
        "preposition_or_particle"
    }

    # Прямой синоним WordNet
    if candidate in get_wordnet_synonyms(target, target_pos):
        return "wordnet_synonym"

    # То же значение в WordNet
    if share_wordnet_synset(candidate, target, target_pos):
        return "same_wordnet_synset"

    # Антоним
    if is_antonym_candidate(candidate, target, target_pos):
        return "wordnet_antonym"

    # Противоположная эмоциональная полярность
    if has_opposite_emotion_polarity(candidate, target):
        return "opposite_emotion_polarity"

    # Слишком высокая векторная близость
    if (
        not skip_high_similarity_filter
        and similarity > MAX_ALLOWED_VECTOR_SIMILARITY
    ):
        return "too_high_vector_similarity"

    return None


df["semantic_candidates_clean"] = df.apply(
    apply_near_equivalent_filter,
    axis=1
)

df["removed_by_near_equivalent_filter"] = df.apply(
    collect_removed_by_near_equivalent_filter,
    axis=1
)

df["num_semantic_candidates_clean"] = df["semantic_candidates_clean"].apply(len)

df.loc[
    df["target_clean"].isin(["however", "out"]),
    [
        "target_clean",
        "semantic_candidates_context",
        "semantic_candidates_clean",
        "removed_by_near_equivalent_filter"
    ]
]

,target_clean,semantic_candidates_context,semantic_candidates_clean,removed_by_near_equivalent_filter
2,however,"[although, though, nevertheless, therefore, mo...","[although, though, therefore, moreover, instead]","[{'candidate': 'nevertheless', 'reason': 'word..."
10,out,"[up, off, back, away, into, down, over, on, fr...","[up, off, back, away, into, down, over, on, fr...",[]
66,however,"[although, nevertheless, therefore, moreover, ...","[although, therefore, moreover, instead]","[{'candidate': 'nevertheless', 'reason': 'word..."
104,however,"[although, though, nevertheless, therefore, mo...","[although, though, therefore, moreover, instead]","[{'candidate': 'nevertheless', 'reason': 'word..."
128,however,"[although, though, nevertheless, therefore, mo...","[although, though, therefore, moreover, instead]","[{'candidate': 'nevertheless', 'reason': 'word..."
198,out,"[off, away, down, over]","[off, away, down, over]",[]
203,however,"[although, nevertheless, therefore, moreover, ...","[although, therefore, moreover, instead]","[{'candidate': 'nevertheless', 'reason': 'word..."
232,out,"[up, off, back, down, from, around]","[up, off, back, down, from, around]",[]
278,however,"[although, though, nevertheless, therefore, mo...","[although, though, therefore, moreover, instead]","[{'candidate': 'nevertheless', 'reason': 'word..."
331,however,"[though, nevertheless, therefore, moreover, in...","[though, therefore, moreover, instead]","[{'candidate': 'nevertheless', 'reason': 'word..."


In [42]:
# =========================
# БЛОК 6. Learner corpus из answer_key
# =========================

def build_learner_errors_from_answer_key(answer_key_df: pd.DataFrame) -> pd.DataFrame:
    """
    Создает learner corpus из реальных дистракторов ЕГЭ.

    Каждая строка превращается в пары:
    target -> distractor_1
    target -> distractor_2
    target -> distractor_3
    """
    rows = []

    for _, row in answer_key_df.iterrows():
        target = clean_text(row.get("target", "")).lower()

        item_key = make_item_key(
            row.get("kod", ""),
            row.get("title", ""),
            row.get("task_id", "")
        )

        for col in [
            "distractor_1",
            "distractor_2",
            "distractor_3"
        ]:
            error_variant = clean_text(row.get(col, "")).lower()

            if not target or not error_variant:
                continue

            if error_variant == target:
                continue

            rows.append(
                {
                    "target": target,
                    "error_variant": error_variant,
                    "item_key": item_key,
                    "source": "EGE_answer_key",
                    "error_type": "real_ege_distractor",
                    "learner_sentence": "",
                    "correct_sentence": "",
                    "comment": "Реальный дистрактор из задания ЕГЭ"
                }
            )

    return pd.DataFrame(rows)

In [43]:
# =========================
# БЛОК 6. Learner corpus из REALEC
# =========================

def build_learner_errors_from_realec(realec_df: pd.DataFrame) -> pd.DataFrame:
    """
    Приводит REALEC к единому формату.

    Используем не только:
    correct_element -> error_variant

    но и контекст:
    learner_sentence
    correct_sentence
    comment
    error_type
    """
    rows = []

    for _, row in realec_df.iterrows():
        target = clean_text(row.get("correct_element", "")).lower()

        if not target:
            target = clean_text(row.get("target_head", "")).lower()

        error_variant = clean_text(row.get("error_variant", "")).lower()

        if not target or not error_variant:
            continue

        if target == error_variant:
            continue

        rows.append(
            {
                "target": target,
                "error_variant": error_variant,
                "item_key": "REALEC",
                "source": clean_text(row.get("source_error", "REALEC")) or "REALEC",
                "error_type": clean_text(row.get("error_type", "learner_error")),
                "learner_sentence": clean_text(row.get("learner_sentence", "")),
                "correct_sentence": clean_text(row.get("correct_sentence", "")),
                "comment": clean_text(row.get("comment", ""))
            }
        )

    return pd.DataFrame(rows)

In [44]:
# =========================
# БЛОК 6. Собираем общий learner corpus
# =========================

ege_learner_errors_df = build_learner_errors_from_answer_key(answer_key_df)
realec_learner_errors_df = build_learner_errors_from_realec(realec_df)

learner_errors_df = pd.concat(
    [
        ege_learner_errors_df,
        realec_learner_errors_df
    ],
    ignore_index=True
).drop_duplicates(
    subset=[
        "target",
        "error_variant",
        "item_key",
        "source"
    ]
).reset_index(drop=True)

print("EGE learner errors:", ege_learner_errors_df.shape)
print("REALEC learner errors:", realec_learner_errors_df.shape)
print("Общий learner_errors_df:", learner_errors_df.shape)

print("\nИсточники:")
print(learner_errors_df["source"].value_counts())

learner_errors_df.head()

EGE learner errors: (1027, 8)
REALEC learner errors: (339, 8)
Общий learner_errors_df: (1131, 8)

Источники:
source
EGE_answer_key    1027
REALEC             104
Name: count, dtype: int64


,target,error_variant,item_key,source,error_type,learner_sentence,correct_sentence,comment
0,announced,promoted,09B13D|Going on a hike|30,EGE_answer_key,real_ege_distractor,,,Реальный дистрактор из задания ЕГЭ
1,announced,extended,09B13D|Going on a hike|30,EGE_answer_key,real_ege_distractor,,,Реальный дистрактор из задания ЕГЭ
2,announced,presented,09B13D|Going on a hike|30,EGE_answer_key,real_ege_distractor,,,Реальный дистрактор из задания ЕГЭ
3,interested,delighted,09B13D|Going on a hike|31,EGE_answer_key,real_ege_distractor,,,Реальный дистрактор из задания ЕГЭ
4,interested,inspired,09B13D|Going on a hike|31,EGE_answer_key,real_ege_distractor,,,Реальный дистрактор из задания ЕГЭ


In [45]:
# =========================
# БЛОК 6. Контекст learner errors
# =========================

def build_learner_context(row) -> str:
    """
    Собирает текстовый контекст ошибки из learner corpus.
    """
    parts = [
        row.get("learner_sentence", ""),
        row.get("correct_sentence", ""),
        row.get("comment", ""),
        row.get("error_type", "")
    ]

    return clean_text(
        " ".join(
            clean_text(part)
            for part in parts
            if clean_text(part)
        )
    )


def get_contextual_learner_candidates(
    target: str,
    current_item_key: str,
    current_sentence: str,
    current_text: str,
    learner_errors_df: pd.DataFrame,
    topn: int = 20
) -> List[str]:
    """
    Возвращает learner candidates с учетом контекста.

    Для EGE answer_key контекста почти нет, поэтому они просто идут как пары.
    Для REALEC считаем похожесть текущего контекста на learner_sentence / correct_sentence.
    """
    target = clean_text(target).lower()

    matched = learner_errors_df[
        (learner_errors_df["target"].astype(str).str.lower().str.strip() == target)
        &
        (learner_errors_df["item_key"].astype(str) != str(current_item_key))
    ].copy()

    if matched.empty:
        return []

    matched["learner_context"] = matched.apply(
        build_learner_context,
        axis=1
    )

    current_context = clean_text(
        current_sentence + " " + current_text
    )

    # Если контекста у matched нет, возвращаем пары target -> error_variant.
    if matched["learner_context"].eq("").all():
        return (
            matched["error_variant"]
            .dropna()
            .astype(str)
            .str.lower()
            .str.strip()
            .drop_duplicates()
            .head(topn)
            .tolist()
        )

    contexts = matched["learner_context"].fillna("").tolist()

    try:
        vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            min_df=1
        )

        matrix = vectorizer.fit_transform(
            [current_context] + contexts
        )

        scores = cosine_similarity(
            matrix[0],
            matrix[1:]
        ).ravel()

    except ValueError:
        scores = np.zeros(len(matched))

    matched["learner_context_score"] = scores

    matched = matched.sort_values(
        by=[
            "learner_context_score",
            "source"
        ],
        ascending=[
            False,
            True
        ]
    )

    return (
        matched["error_variant"]
        .dropna()
        .astype(str)
        .str.lower()
        .str.strip()
        .drop_duplicates()
        .head(topn)
        .tolist()
    )

In [46]:
# =========================
# БЛОК 6. Добавляем learner candidates к df
# =========================

df["learner_candidates_raw"] = df.apply(
    lambda row: get_contextual_learner_candidates(
        target=row["target_clean"],
        current_item_key=row["item_key"],
        current_sentence=row["sentence_clean"],
        current_text=row["text_clean"],
        learner_errors_df=learner_errors_df,
        topn=20
    ),
    axis=1
)

df[
    [
        "target_clean",
        "learner_candidates_raw"
    ]
].head(20)

,target_clean,learner_candidates_raw
0,announced,"[extended, promoted, presented]"
1,interested,[]
2,however,"[despite, otherwise, likewise, although, moreo..."
3,thick,[]
4,remind,"[revise, review, recall, remember]"
5,observing,[]
6,told,"[talked, spoke, said]"
7,formed,[]
8,introduced,"[proposed, announced, welcomed]"
9,attracted,[]


In [47]:
# =========================
# БЛОК 6. Фильтрация learner candidates
# =========================

def filter_learner_candidates(row) -> List[str]:
    """
    Применяет к learner candidates уже созданные фильтры:
    - грамматика;
    - контекст;
    - почти-эквиваленты.
    """
    result = []

    for candidate in row["learner_candidates_raw"]:
        if not is_grammar_compatible(candidate, row):
            continue

        if context_filter_reason(candidate, row) is not None:
            continue

        if near_equivalent_reason(candidate, row) is not None:
            continue

        result.append(candidate)

    return unique_list(result)


df["learner_candidates_clean"] = df.apply(
    filter_learner_candidates,
    axis=1
)

df[
    [
        "target_clean",
        "semantic_candidates_clean",
        "learner_candidates_raw",
        "learner_candidates_clean"
    ]
].head(20)

,target_clean,semantic_candidates_clean,learner_candidates_raw,learner_candidates_clean
0,announced,"[agreed, planned, decided, expected, unveiled,...","[extended, promoted, presented]","[extended, promoted, presented]"
1,interested,"[involved, keen, realized]",[],[]
2,however,"[although, though, therefore, moreover, instead]","[despite, otherwise, likewise, although, moreo...","[otherwise, although, moreover, therefore]"
3,thick,"[tall, dark, yellow, dry, shiny, rough, gray, ...",[],[]
4,remind,"[tell, ask, forget, wish, inform, remember, im...","[revise, review, recall, remember]","[revise, review, recall, remember]"
5,observing,"[monitoring, conducting, analyzing, photograph...",[],[]
6,told,"[quoted, insisted, spoke, informed, explained,...","[talked, spoke, said]","[talked, spoke]"
7,formed,"[merged, established, joined, created, founded...",[],[]
8,introduced,"[adopted, created, modified, used, changed, pr...","[proposed, announced, welcomed]","[proposed, announced, welcomed]"
9,attracted,"[garnered, encouraged, generated, grew, grown,...",[],[]


In [48]:
df[
    [
        "target_clean",
        "semantic_candidates_clean",
        "learner_candidates_raw",
        "learner_candidates_clean"
    ]
].head(20)

,target_clean,semantic_candidates_clean,learner_candidates_raw,learner_candidates_clean
0,announced,"[agreed, planned, decided, expected, unveiled,...","[extended, promoted, presented]","[extended, promoted, presented]"
1,interested,"[involved, keen, realized]",[],[]
2,however,"[although, though, therefore, moreover, instead]","[despite, otherwise, likewise, although, moreo...","[otherwise, although, moreover, therefore]"
3,thick,"[tall, dark, yellow, dry, shiny, rough, gray, ...",[],[]
4,remind,"[tell, ask, forget, wish, inform, remember, im...","[revise, review, recall, remember]","[revise, review, recall, remember]"
5,observing,"[monitoring, conducting, analyzing, photograph...",[],[]
6,told,"[quoted, insisted, spoke, informed, explained,...","[talked, spoke, said]","[talked, spoke]"
7,formed,"[merged, established, joined, created, founded...",[],[]
8,introduced,"[adopted, created, modified, used, changed, pr...","[proposed, announced, welcomed]","[proposed, announced, welcomed]"
9,attracted,"[garnered, encouraged, generated, grew, grown,...",[],[]


In [49]:
# =========================
# БЛОК 7. Настройки ранжирования
# =========================

# Если True, используем реальные дистракторы текущего задания.
# Это полезно для демонстрации качества на размеченном корпусе.
# Если нужна честная проверка генерации без подсказки, поставь False.
USE_CURRENT_GOLD_EGE_DISTRACTORS = False

# Бонусы за источник кандидата
SOURCE_BONUSES = {
    "SemanticModel": 0.05,
    "LearnerCorpus": 0.25,
    "CurrentGoldEGE": 0.45
}

# Слишком высокая близость опасна только для чисто семантических кандидатов.
# Для learner / gold кандидатов это не штрафуем.
HIGH_SIMILARITY_PENALTY = 0.15
HIGH_SIMILARITY_THRESHOLD = 0.78

In [50]:
# =========================
# БЛОК 7. Объединяем кандидатов с источниками
# =========================

def add_candidate_to_pool(pool: Dict[str, Dict], candidate: str, source: str):
    """
    Добавляет кандидата в общий пул.

    Если кандидат уже есть, просто добавляет новый источник.
    """
    candidate = clean_text(candidate).lower()

    if not candidate:
        return

    if candidate not in pool:
        pool[candidate] = {
            "candidate": candidate,
            "sources": []
        }

    if source not in pool[candidate]["sources"]:
        pool[candidate]["sources"].append(source)


def build_candidate_pool(row) -> List[Dict]:
    """
    Собирает кандидатов из всех источников:
    - semantic_candidates_clean
    - learner_candidates_clean
    - gold_distractors, если включен режим USE_CURRENT_GOLD_EGE_DISTRACTORS
    """
    pool = {}

    for candidate in row["semantic_candidates_clean"]:
        add_candidate_to_pool(
            pool,
            candidate,
            "SemanticModel"
        )

    for candidate in row["learner_candidates_clean"]:
        add_candidate_to_pool(
            pool,
            candidate,
            "LearnerCorpus"
        )

    if USE_CURRENT_GOLD_EGE_DISTRACTORS:
        for candidate in row["gold_distractors"]:
            add_candidate_to_pool(
                pool,
                candidate,
                "CurrentGoldEGE"
            )

    return list(pool.values())

In [51]:
# =========================
# БЛОК 7. Скоринг кандидата
# =========================

def score_candidate(candidate_record: Dict, row) -> Dict:
    """
    Считает итоговый score кандидата.

    Компоненты:
    - GloVe similarity к target;
    - бонус за источник;
    - штраф за слишком высокую близость, если кандидат только из SemanticModel.
    """
    candidate = candidate_record["candidate"]
    sources = candidate_record["sources"]
    target = row["target_clean"]
    target_group = row["target_form_group"]

    similarity = vector_similarity(
        target,
        candidate,
        glove_model
    )

    source_bonus = sum(
        SOURCE_BONUSES.get(source, 0.0)
        for source in sources
    )

    penalty = 0.0

    protected_sources = {
        "LearnerCorpus",
        "CurrentGoldEGE"
    }

    is_protected = bool(
        set(sources) & protected_sources
    )

    # Для discourse markers и particles высокая близость допустима.
    is_function_word_group = target_group in {
        "discourse_marker",
        "preposition_or_particle"
    }

    if (
        not is_protected
        and not is_function_word_group
        and similarity > HIGH_SIMILARITY_THRESHOLD
    ):
        penalty = HIGH_SIMILARITY_PENALTY

    final_score = similarity + source_bonus - penalty

    return {
        "candidate": candidate,
        "sources": sources,
        "similarity": similarity,
        "source_bonus": source_bonus,
        "penalty": penalty,
        "final_score": final_score
    }

In [52]:
# =========================
# БЛОК 7. Ранжирование кандидатов
# =========================

def rank_candidates_for_row(row) -> List[Dict]:
    """
    Собирает и ранжирует кандидатов для одной строки.
    """
    pool = build_candidate_pool(row)

    scored = [
        score_candidate(candidate_record, row)
        for candidate_record in pool
    ]

    scored = sorted(
        scored,
        key=lambda x: x["final_score"],
        reverse=True
    )

    return scored


df["ranked_candidates"] = df.apply(
    rank_candidates_for_row,
    axis=1
)

df["ranked_candidate_words"] = df["ranked_candidates"].apply(
    lambda ranked: [
        item["candidate"]
        for item in ranked
    ]
)

df[
    [
        "target_clean",
        "semantic_candidates_clean",
        "learner_candidates_clean",
        "gold_distractors",
        "ranked_candidate_words"
    ]
].head(20)

,target_clean,semantic_candidates_clean,learner_candidates_clean,gold_distractors,ranked_candidate_words
0,announced,"[agreed, planned, decided, expected, unveiled,...","[extended, promoted, presented]","[promoted, extended, presented]","[presented, agreed, planned, decided, extended..."
1,interested,"[involved, keen, realized]",[],"[delighted, inspired, excited]","[involved, keen, realized]"
2,however,"[although, though, therefore, moreover, instead]","[otherwise, although, moreover, therefore]","[moreover, although, otherwise]","[although, therefore, moreover, though, otherw..."
3,thick,"[tall, dark, yellow, dry, shiny, rough, gray, ...",[],"[heavy, strong, rich]","[tall, dark, yellow, dry, shiny, rough, gray, ..."
4,remind,"[tell, ask, forget, wish, inform, remember, im...","[revise, review, recall, remember]","[review, remember, reflect]","[remember, tell, ask, forget, wish, inform, im..."
5,observing,"[monitoring, conducting, analyzing, photograph...",[],"[obtaining, offending, objecting]","[monitoring, conducting, analyzing, photograph..."
6,told,"[quoted, insisted, spoke, informed, explained,...","[talked, spoke]","[talked, spoke, said]","[spoke, talked, quoted, insisted, informed, ex..."
7,formed,"[merged, established, joined, created, founded...",[],"[kept, held, used]","[merged, established, joined, created, founded..."
8,introduced,"[adopted, created, modified, used, changed, pr...","[proposed, announced, welcomed]","[called, greeted, met]","[proposed, announced, adopted, created, modifi..."
9,attracted,"[garnered, encouraged, generated, grew, grown,...",[],"[involved, appealed, engaged]","[garnered, encouraged, generated, grew, grown,..."


In [53]:
# =========================
# БЛОК 7. Диагностика ранжирования
# =========================

for _, row in df.head(12).iterrows():
    print("=" * 100)
    print("target:", row["target_clean"])
    print("gold:", row["gold_distractors"])
    print("semantic:", row["semantic_candidates_clean"][:8])
    print("learner:", row["learner_candidates_clean"][:8])
    print("ranked top-10:")

    for item in row["ranked_candidates"][:10]:
        print(
            item["candidate"],
            "| score:",
            round(item["final_score"], 4),
            "| sim:",
            round(item["similarity"], 4),
            "| sources:",
            item["sources"]
        )

target: announced
gold: ['promoted', 'extended', 'presented']
semantic: ['agreed', 'planned', 'decided', 'expected', 'unveiled', 'proposed', 'confirmed', 'approved']
learner: ['extended', 'promoted', 'presented']
ranked top-10:
presented | score: 0.8601 | sim: 0.6101 | sources: ['LearnerCorpus']
agreed | score: 0.8209 | sim: 0.7709 | sources: ['SemanticModel']
planned | score: 0.7989 | sim: 0.7489 | sources: ['SemanticModel']
decided | score: 0.7955 | sim: 0.7455 | sources: ['SemanticModel']
extended | score: 0.7869 | sim: 0.5369 | sources: ['LearnerCorpus']
expected | score: 0.783 | sim: 0.733 | sources: ['SemanticModel']
unveiled | score: 0.7611 | sim: 0.7111 | sources: ['SemanticModel']
proposed | score: 0.7497 | sim: 0.6997 | sources: ['SemanticModel']
confirmed | score: 0.7478 | sim: 0.6978 | sources: ['SemanticModel']
approved | score: 0.7446 | sim: 0.6946 | sources: ['SemanticModel']
target: interested
gold: ['delighted', 'inspired', 'excited']
semantic: ['involved', 'keen', 're

In [54]:
# =========================
# БЛОК 8. Загрузка CEFR vocabulary profile
# =========================

CEFR_URL = "https://raw.githubusercontent.com/openlanguageprofiles/olp-en-cefrj/master/cefrj-vocabulary-profile-1.5.csv"

cefr_df = pd.read_csv(CEFR_URL)

print("CEFR dataframe:", cefr_df.shape)
print(cefr_df.columns.tolist())

cefr_df.head()

CEFR dataframe: (7799, 6)
['headword', 'pos', 'CEFR', 'CoreInventory 1', 'CoreInventory 2', 'Threshold']


,headword,pos,CEFR,CoreInventory 1,CoreInventory 2,Threshold
0,a,determiner,A1,NaN,NaN,NaN
1,a.m./A.M./am/AM,adverb,A1,NaN,NaN,NaN
2,abandon,verb,B1,NaN,NaN,NaN
3,abandoned,adjective,B2,NaN,NaN,NaN
4,ability,noun,A2,NaN,NaN,NaN


In [55]:
# =========================
# БЛОК 8. Создаем словарь word -> CEFR
# =========================

CEFR_ORDER = {
    "A1": 1,
    "A2": 2,
    "B1": 3,
    "B2": 4,
    "C1": 5,
    "C2": 6
}

# Ручные исправления для слов, которые CEFR-J может не распознать
CEFR_OVERRIDES = {
    "announced": "B1",
    "presented": "A1",
    "extended": "B1",
    "promoted": "B1",
    "inspired": "B1",
    "interested": "A1",
    "delighted": "B1",
    "excited": "A1",
    "however": "A2",
    "therefore": "B1",
    "moreover": "B1",
    "otherwise": "B1",
    "meanwhile": "B1"
}


def detect_cefr_columns(cefr_df: pd.DataFrame) -> Tuple[str, str]:
    """
    Автоматически определяет, где в таблице слово и где уровень CEFR.
    """
    possible_word_columns = [
        "headword",
        "word",
        "lemma",
        "entry"
    ]

    possible_level_columns = [
        "CEFR",
        "cefr",
        "level",
        "Level"
    ]

    word_column = None
    level_column = None

    for col in possible_word_columns:
        if col in cefr_df.columns:
            word_column = col
            break

    for col in possible_level_columns:
        if col in cefr_df.columns:
            level_column = col
            break

    if word_column is None:
        word_column = cefr_df.columns[0]

    if level_column is None:
        level_column = cefr_df.columns[-1]

    return word_column, level_column


word_column, level_column = detect_cefr_columns(cefr_df)

print("word_column:", word_column)
print("level_column:", level_column)


cefr_level_map = {}

for _, row in cefr_df.iterrows():
    word = clean_text(row[word_column]).lower()
    level = clean_text(row[level_column]).upper()[:2]

    if not word:
        continue

    if level not in CEFR_ORDER:
        continue

    # Если слово встречается несколько раз, берем самый простой уровень.
    if (
        word not in cefr_level_map
        or CEFR_ORDER[level] < CEFR_ORDER[cefr_level_map[word]]
    ):
        cefr_level_map[word] = level


cefr_level_map.update(CEFR_OVERRIDES)

print("Количество слов в CEFR map:", len(cefr_level_map))

word_column: headword
level_column: CEFR
Количество слов в CEFR map: 6868


In [56]:
# =========================
# БЛОК 8. Функция определения CEFR уровня
# =========================

def get_cefr_level(word: str) -> Optional[str]:
    """
    Возвращает CEFR-уровень слова.

    Если точная форма не найдена, пробуем:
    - лемму;
    - части словосочетания.
    """
    word = clean_text(word).lower()

    if not word:
        return None

    if word in cefr_level_map:
        return cefr_level_map[word]

    lemma = get_candidate_lemma(word)

    if lemma in cefr_level_map:
        return cefr_level_map[lemma]

    # Для словосочетаний берем максимальный уровень частей.
    part_levels = []

    for part in word.split():
        part = clean_text(part).lower()

        if part in cefr_level_map:
            part_levels.append(cefr_level_map[part])
            continue

        part_lemma = get_candidate_lemma(part)

        if part_lemma in cefr_level_map:
            part_levels.append(cefr_level_map[part_lemma])

    if part_levels:
        return max(
            part_levels,
            key=lambda level: CEFR_ORDER[level]
        )

    return None

In [57]:
# =========================
# БЛОК 8. Проверка CEFR совместимости
# =========================

MIN_CEFR_LEVEL = "A1"
MAX_CEFR_LEVEL = "B2"
KEEP_UNKNOWN_CEFR = True


def is_cefr_compatible(candidate: str, row) -> bool:
    """
    Проверяет, подходит ли кандидат по уровню CEFR.

    Логика:
    - выше B2 отсекаем;
    - unknown можно оставить, но отправить на диагностику;
    - если target известен, не берем кандидата намного сложнее target.
    """
    candidate_level = get_cefr_level(candidate)
    target_level = get_cefr_level(row["target_clean"])

    if candidate_level is None:
        return KEEP_UNKNOWN_CEFR

    candidate_rank = CEFR_ORDER[candidate_level]

    if candidate_rank < CEFR_ORDER[MIN_CEFR_LEVEL]:
        return False

    if candidate_rank > CEFR_ORDER[MAX_CEFR_LEVEL]:
        return False

    if target_level is not None:
        target_rank = CEFR_ORDER[target_level]

        # Кандидат не должен быть больше чем на 2 уровня сложнее target.
        if candidate_rank - target_rank > 2:
            return False

    return True

In [58]:
# =========================
# БЛОК 8. Применяем CEFR к ranked_candidates
# =========================

def apply_cefr_to_ranked_candidates(row) -> List[Dict]:
    """
    Фильтрует уже ранжированные кандидаты по CEFR.
    """
    result = []

    for item in row["ranked_candidates"]:
        candidate = item["candidate"]

        if is_cefr_compatible(candidate, row):
            new_item = item.copy()
            new_item["candidate_cefr"] = get_cefr_level(candidate)
            new_item["target_cefr"] = get_cefr_level(row["target_clean"])
            result.append(new_item)

    return result


df["ranked_candidates_cefr"] = df.apply(
    apply_cefr_to_ranked_candidates,
    axis=1
)

df["ranked_candidate_words_cefr"] = df["ranked_candidates_cefr"].apply(
    lambda ranked: [
        item["candidate"]
        for item in ranked
    ]
)

df[
    [
        "target_clean",
        "ranked_candidate_words",
        "ranked_candidate_words_cefr"
    ]
].head(20)

,target_clean,ranked_candidate_words,ranked_candidate_words_cefr
0,announced,"[presented, agreed, planned, decided, extended...","[presented, agreed, planned, decided, extended..."
1,interested,"[involved, keen, realized]","[involved, realized]"
2,however,"[although, therefore, moreover, though, otherw...","[although, therefore, moreover, though, otherw..."
3,thick,"[tall, dark, yellow, dry, shiny, rough, gray, ...","[tall, dark, yellow, dry, shiny, rough, gray, ..."
4,remind,"[remember, tell, ask, forget, wish, inform, im...","[remember, tell, ask, forget, wish, inform, im..."
5,observing,"[monitoring, conducting, analyzing, photograph...","[monitoring, conducting, analyzing, photograph..."
6,told,"[spoke, talked, quoted, insisted, informed, ex...","[spoke, talked, insisted, informed, explained,..."
7,formed,"[merged, established, joined, created, founded...","[established, joined, created, incorporated, b..."
8,introduced,"[proposed, announced, adopted, created, modifi...","[announced, created, modified, used, changed, ..."
9,attracted,"[garnered, encouraged, generated, grew, grown,...","[garnered, encouraged, generated, grew, grown,..."


In [59]:
# =========================
# БЛОК 8. Диагностика CEFR
# =========================

for _, row in df.head(12).iterrows():
    print("=" * 100)
    print("target:", row["target_clean"])
    print("target_cefr:", get_cefr_level(row["target_clean"]))
    print("ranked after CEFR:")

    for item in row["ranked_candidates_cefr"][:10]:
        print(
            item["candidate"],
            "| cefr:",
            item["candidate_cefr"],
            "| score:",
            round(item["final_score"], 4),
            "| sources:",
            item["sources"]
        )

target: announced
target_cefr: B1
ranked after CEFR:
presented | cefr: A1 | score: 0.8601 | sources: ['LearnerCorpus']
agreed | cefr: A1 | score: 0.8209 | sources: ['SemanticModel']
planned | cefr: A1 | score: 0.7989 | sources: ['SemanticModel']
decided | cefr: A2 | score: 0.7955 | sources: ['SemanticModel']
extended | cefr: B1 | score: 0.7869 | sources: ['LearnerCorpus']
expected | cefr: B2 | score: 0.783 | sources: ['SemanticModel']
unveiled | cefr: None | score: 0.7611 | sources: ['SemanticModel']
proposed | cefr: B2 | score: 0.7497 | sources: ['SemanticModel']
confirmed | cefr: B1 | score: 0.7478 | sources: ['SemanticModel']
approved | cefr: B1 | score: 0.7446 | sources: ['SemanticModel']
target: interested
target_cefr: A1
ranked after CEFR:
involved | cefr: B1 | score: 0.739 | sources: ['SemanticModel']
realized | cefr: None | score: 0.6538 | sources: ['SemanticModel']
target: however
target_cefr: A2
ranked after CEFR:
although | cefr: A2 | score: 1.2658 | sources: ['SemanticModel

In [60]:
# =========================
# БЛОК 9. Расширенный контекст текста
# =========================

def split_into_sentences(text: str) -> List[str]:
    """
    Делит полный текст на предложения через spaCy.
    """
    text = clean_text(text)

    if not text:
        return []

    doc = nlp(text)

    return [
        clean_text(sent.text)
        for sent in doc.sents
        if clean_text(sent.text)
    ]


def token_set_for_matching(text: str) -> set:
    """
    Делает набор лемм для мягкого сопоставления предложений.

    Это нужно на случай, если sentence_with_answer
    немного отличается от предложения внутри full_text.
    """
    doc = nlp(clean_text(text).lower())

    return {
        token.lemma_
        for token in doc
        if token.is_alpha and token.text not in stop_words
    }


def find_target_sentence_index(
    sentences: List[str],
    sentence_with_answer: str,
    target: str
) -> int:
    """
    Находит индекс предложения с target внутри полного текста.

    Сначала ищем target как отдельное слово.
    Если таких предложений несколько, выбираем то, которое больше всего
    похоже на sentence_with_answer по пересечению лемм.
    """
    if not sentences:
        return -1

    target = clean_text(target).lower()
    answer_tokens = token_set_for_matching(sentence_with_answer)

    best_index = -1
    best_score = -1

    for i, sentence in enumerate(sentences):
        sentence_clean = clean_text(sentence)
        sentence_lower = sentence_clean.lower()

        has_target = bool(
            re.search(
                r"\b" + re.escape(target) + r"\b",
                sentence_lower,
                flags=re.IGNORECASE
            )
        )

        sentence_tokens = token_set_for_matching(sentence_clean)
        overlap = len(answer_tokens & sentence_tokens)

        # target важнее, чем просто лексическое пересечение
        score = overlap + (10 if has_target else 0)

        if score > best_score:
            best_score = score
            best_index = i

    return best_index


def get_extended_context(
    full_text: str,
    sentence_with_answer: str,
    target: str,
    window: int = 1
) -> str:
    """
    Возвращает расширенный контекст:
    предложение с target + window предложений слева и справа.
    """
    sentences = split_into_sentences(full_text)

    if not sentences:
        return clean_text(sentence_with_answer)

    target_index = find_target_sentence_index(
        sentences=sentences,
        sentence_with_answer=sentence_with_answer,
        target=target
    )

    if target_index == -1:
        return clean_text(sentence_with_answer)

    start = max(0, target_index - window)
    end = min(len(sentences), target_index + window + 1)

    return clean_text(
        " ".join(sentences[start:end])
    )

In [61]:
# =========================
# БЛОК 9. Применяем расширенный контекст
# =========================

CONTEXT_WINDOW = 1

df["extended_context"] = df.apply(
    lambda row: get_extended_context(
        full_text=row["text_clean"],
        sentence_with_answer=row["sentence_clean"],
        target=row["target_clean"],
        window=CONTEXT_WINDOW
    ),
    axis=1
)

df["extended_context_length_words"] = df["extended_context"].apply(
    lambda text: len(clean_text(text).split())
)

df[
    [
        "target_clean",
        "sentence_clean",
        "extended_context",
        "extended_context_length_words"
    ]
].head(10)

,target_clean,sentence_clean,extended_context,extended_context_length_words
0,announced,"Last week, our coach Mr Brown announced we wer...","Occasionally, the team took field trips to pla...",57
1,interested,We weren't all that interested in the outdoors.,"Simon is my best friend, and the two of us had...",49
2,however,"However, Mr Brown claimed that we'd really enj...",We weren't all that interested in the outdoors...,40
3,thick,"Now we were in a state park, marching through ...",Simon and I were sure we'd rather be back home...,59
4,remind,"Doesn't its shape remind you of an elephant?"" ...","I looked around, shouting, ""Check out that tre...",21
5,observing,"You don't see stuff like that in the city, now...","You don't see stuff like that in the city, now...",48
6,told,"Out here, away from the city, I was starting t...",We were beginning to see the amazing things ar...,33
7,formed,I formed the habit now of going every Saturday...,"I never took Olivia to the theatre, but it was...",38
8,introduced,It was George who introduced me to Olivia Nelson.,I formed the habit now of going every Saturday...,42
9,attracted,"She was not very beautiful but she was tall, v...","She was not very beautiful but she was tall, v...",29


In [62]:
# =========================
# БЛОК 9. Диагностика расширенного контекста
# =========================

for _, row in df.head(8).iterrows():
    print("=" * 100)
    print("target:", row["target_clean"])
    print("sentence:")
    print(row["sentence_clean"])
    print("\nextended_context:")
    print(row["extended_context"])
    print("words:", row["extended_context_length_words"])

target: announced
sentence:
Last week, our coach Mr Brown announced we were going on a hike through a forest in a state park.

extended_context:
Occasionally, the team took field trips to places outside of our neighbourhood on the weekends. Last week, our coach Mr Brown announced we were going on a hike through a forest in a state park. Simon is my best friend, and the two of us had both liked the trips to the museums, plays, and basketball games.
words: 57
target: interested
sentence:
We weren't all that interested in the outdoors.

extended_context:
Simon is my best friend, and the two of us had both liked the trips to the museums, plays, and basketball games. We weren't all that interested in the outdoors. However, Mr Brown claimed that we'd really enjoy discovering how amazing nature can be-especially for the big city kids.
words: 49
target: however
sentence:
However, Mr Brown claimed that we'd really enjoy discovering how amazing nature can be-especially for the big city kids.

ex

In [63]:
# =========================
# БЛОК 10. Настройки BERT-фильтра
# =========================

USE_BERT_FILTER = True

BERT_MODEL_NAME = "distilroberta-base"

# Если кандидат имеет вероятность выше этой доли от правильного ответа,
# он считается слишком подходящим к контексту.
BERT_MAX_RELATIVE_TO_TARGET = 0.45

# Если BERT вообще почти не знает это слово в контексте, оставляем его.
# Поэтому нижний порог не делаем слишком жестким.
BERT_MIN_CANDIDATE_SCORE = 0.0

print("USE_BERT_FILTER:", USE_BERT_FILTER)

USE_BERT_FILTER: True


In [64]:
# =========================
# БЛОК 10. Загрузка masked language model
# =========================

if USE_BERT_FILTER:
    from transformers import pipeline

    fill_mask = pipeline(
        "fill-mask",
        model=BERT_MODEL_NAME
    )

    print("BERT model loaded:", BERT_MODEL_NAME)
else:
    fill_mask = None
    print("BERT filter выключен.")

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

BERT model loaded: distilroberta-base


In [65]:
# =========================
# БЛОК 10. Маскировка target в extended_context
# =========================

def mask_target_in_context(
    context: str,
    target: str,
    mask_token: str = "<mask>"
) -> str:
    """
    Заменяет первое вхождение target в extended_context на <mask>.
    """
    context = clean_text(context)
    target = clean_text(target)

    pattern = r"\b" + re.escape(target) + r"\b"

    masked = re.sub(
        pattern,
        mask_token,
        context,
        count=1,
        flags=re.IGNORECASE
    )

    # Если target почему-то не найден, добавляем mask перед контекстом.
    if masked == context:
        masked = f"{mask_token} {context}"

    return masked


def shorten_masked_context(
    masked_context: str,
    mask_token: str = "<mask>",
    window_words: int = 80
) -> str:
    """
    Обрезает слишком длинный контекст вокруг <mask>,
    чтобы BERT не получал чрезмерно длинный ввод.
    """
    tokens = clean_text(masked_context).split()

    if mask_token not in tokens:
        return clean_text(masked_context)

    mask_index = tokens.index(mask_token)

    start = max(
        0,
        mask_index - window_words
    )

    end = min(
        len(tokens),
        mask_index + window_words + 1
    )

    return " ".join(tokens[start:end])


df["masked_extended_context"] = df.apply(
    lambda row: shorten_masked_context(
        mask_target_in_context(
            row["extended_context"],
            row["target_clean"]
        )
    ),
    axis=1
)

df[
    [
        "target_clean",
        "masked_extended_context"
    ]
].head()

,target_clean,masked_extended_context
0,announced,"Occasionally, the team took field trips to pla..."
1,interested,"Simon is my best friend, and the two of us had..."
2,however,We weren't all that interested in the outdoors...
3,thick,Simon and I were sure we'd rather be back home...
4,remind,"I looked around, shouting, ""Check out that tre..."


In [66]:
# =========================
# БЛОК 10. BERT score
# =========================

def bert_candidate_score(masked_context: str, candidate: str) -> float:
    """
    Возвращает вероятность candidate в позиции <mask>.
    Для multi-word candidates возвращаем 0.0, потому что fill-mask
    работает с одним mask token.
    """
    if not USE_BERT_FILTER:
        return 0.0

    candidate = clean_text(candidate).lower()

    if not candidate:
        return 0.0

    if len(candidate.split()) > 1:
        return 0.0

    try:
        result = fill_mask(
            masked_context,
            targets=[candidate]
        )

        if isinstance(result, list) and result:
            return float(
                result[0].get(
                    "score",
                    0.0
                )
            )

    except Exception:
        return 0.0

    return 0.0

In [67]:
# =========================
# БЛОК 10. Сравнение кандидата с правильным ответом
# =========================

def compute_bert_scores_for_row(row) -> Dict:
    """
    Считает BERT-score:
    - для target;
    - для каждого кандидата после CEFR-фильтра.
    """
    masked_context = row["masked_extended_context"]

    target = row["target_clean"]

    target_score = bert_candidate_score(
        masked_context,
        target
    )

    # Чтобы не делить на 0
    target_score = max(
        target_score,
        1e-12
    )

    candidate_scores = {}

    for item in row["ranked_candidates_cefr"]:
        candidate = item["candidate"]

        candidate_score = bert_candidate_score(
            masked_context,
            candidate
        )

        candidate_scores[candidate] = {
            "candidate_score": candidate_score,
            "target_score": target_score,
            "relative_to_target": candidate_score / target_score
        }

    return {
        "target_score": target_score,
        "candidate_scores": candidate_scores
    }


if USE_BERT_FILTER:
    df["bert_scores"] = df.apply(
        compute_bert_scores_for_row,
        axis=1
    )
else:
    df["bert_scores"] = [
        {
            "target_score": None,
            "candidate_scores": {}
        }
        for _ in range(len(df))
    ]

print("BERT scores computed.")

The specified target token `presented` does not exist in the model vocabulary. Replacing with `present`.
The specified target token `agreed` does not exist in the model vocabulary. Replacing with `ag`.
The specified target token `decided` does not exist in the model vocabulary. Replacing with `dec`.
The specified target token `extended` does not exist in the model vocabulary. Replacing with `ext`.
The specified target token `unveiled` does not exist in the model vocabulary. Replacing with `un`.
The specified target token `proposed` does not exist in the model vocabulary. Replacing with `prop`.
The specified target token `promised` does not exist in the model vocabulary. Replacing with `prom`.
The specified target token `scheduled` does not exist in the model vocabulary. Replacing with `sc`.
The specified target token `rejected` does not exist in the model vocabulary. Replacing with `re`.
The specified target token `launched` does not exist in the model vocabulary. Replacing with `laun`

BERT scores computed.


In [68]:
# =========================
# БЛОК 10. Диагностика BERT-score
# =========================

for _, row in df.head(8).iterrows():
    print("=" * 100)
    print("target:", row["target_clean"])
    print("masked context:")
    print(row["masked_extended_context"])
    print("target_score:", row["bert_scores"]["target_score"])

    print("candidate scores:")

    for item in row["ranked_candidates_cefr"][:8]:
        candidate = item["candidate"]

        scores = row["bert_scores"]["candidate_scores"].get(
            candidate,
            {}
        )

        print(
            candidate,
            "| cand_score:",
            round(scores.get("candidate_score", 0.0), 8),
            "| relative:",
            round(scores.get("relative_to_target", 0.0), 4),
            "| sources:",
            item["sources"]
        )

target: announced
masked context:
Occasionally, the team took field trips to places outside of our neighbourhood on the weekends. Last week, our coach Mr Brown <mask> we were going on a hike through a forest in a state park. Simon is my best friend, and the two of us had both liked the trips to the museums, plays, and basketball games.
target_score: 0.0006681207451038063
candidate scores:
presented | cand_score: 0.0 | relative: 0.0 | sources: ['LearnerCorpus']
agreed | cand_score: 6e-08 | relative: 0.0001 | sources: ['SemanticModel']
planned | cand_score: 0.0 | relative: 0.0 | sources: ['SemanticModel']
decided | cand_score: 0.0 | relative: 0.0 | sources: ['SemanticModel']
extended | cand_score: 0.0 | relative: 0.0 | sources: ['LearnerCorpus']
expected | cand_score: 0.0 | relative: 0.0 | sources: ['SemanticModel']
unveiled | cand_score: 0.0 | relative: 0.0 | sources: ['SemanticModel']
proposed | cand_score: 0.0 | relative: 0.0 | sources: ['SemanticModel']
target: interested
masked cont

In [69]:
# =========================
# БЛОК 10.1. Надежный BERT-фильтр
# =========================

BERT_MIN_RELIABLE_TARGET_SCORE = 1e-7
BERT_MAX_RELATIVE_TO_TARGET = 0.45

PROTECTED_BERT_SOURCES = {
    "CurrentGoldEGE",
    "LearnerCorpus"
}


def bert_filter_reason(item: Dict, row) -> Optional[str]:
    """
    Возвращает причину удаления кандидата через BERT.

    BERT-фильтр применяем осторожно:
    - не удаляем gold / learner candidates;
    - не используем relative score, если target_score слишком маленький;
    - удаляем только кандидаты, которые почти так же вероятны,
      как правильный ответ в расширенном контексте.
    """
    candidate = item["candidate"]
    sources = set(item["sources"])

    # Реальные дистракторы и learner errors не трогаем BERT-фильтром.
    if sources & PROTECTED_BERT_SOURCES:
        return None

    if not USE_BERT_FILTER:
        return None

    bert_info = row["bert_scores"]
    target_score = bert_info["target_score"]

    candidate_info = bert_info["candidate_scores"].get(candidate)

    if candidate_info is None:
        return None

    candidate_score = candidate_info["candidate_score"]
    relative_score = candidate_info["relative_to_target"]

    # Если BERT сам почти не видит правильный ответ,
    # relative score становится ненадежным.
    if target_score < BERT_MIN_RELIABLE_TARGET_SCORE:
        return None

    if candidate_score >= BERT_MIN_CANDIDATE_SCORE and relative_score > BERT_MAX_RELATIVE_TO_TARGET:
        return "too_plausible_in_bert_context"

    return None


def apply_bert_filter_to_ranked(row) -> List[Dict]:
    """
    Применяет BERT-фильтр к ranked_candidates_cefr.
    """
    result = []

    for item in row["ranked_candidates_cefr"]:
        reason = bert_filter_reason(item, row)

        if reason is None:
            result.append(item)

    return result


def collect_removed_by_bert(row) -> List[Dict]:
    """
    Сохраняет кандидаты, удаленные BERT-фильтром.
    """
    removed = []

    for item in row["ranked_candidates_cefr"]:
        reason = bert_filter_reason(item, row)

        if reason is not None:
            candidate = item["candidate"]
            candidate_info = row["bert_scores"]["candidate_scores"].get(candidate, {})

            removed.append(
                {
                    "candidate": candidate,
                    "reason": reason,
                    "candidate_score": candidate_info.get("candidate_score"),
                    "target_score": candidate_info.get("target_score"),
                    "relative_to_target": candidate_info.get("relative_to_target"),
                    "sources": item["sources"]
                }
            )

    return removed


df["ranked_candidates_bert"] = df.apply(
    apply_bert_filter_to_ranked,
    axis=1
)

df["removed_by_bert_filter"] = df.apply(
    collect_removed_by_bert,
    axis=1
)

df["ranked_candidate_words_bert"] = df["ranked_candidates_bert"].apply(
    lambda ranked: [
        item["candidate"]
        for item in ranked
    ]
)

df[
    [
        "target_clean",
        "ranked_candidate_words_cefr",
        "ranked_candidate_words_bert",
        "removed_by_bert_filter"
    ]
].head(10)

,target_clean,ranked_candidate_words_cefr,ranked_candidate_words_bert,removed_by_bert_filter
0,announced,"[presented, agreed, planned, decided, extended...","[presented, agreed, planned, decided, extended...",[]
1,interested,"[involved, realized]","[involved, realized]",[]
2,however,"[although, therefore, moreover, though, otherw...","[although, therefore, moreover, though, otherw...",[]
3,thick,"[tall, dark, yellow, dry, shiny, rough, gray, ...","[tall, dark, yellow, dry, shiny, rough, gray, ...",[]
4,remind,"[remember, tell, ask, forget, wish, inform, im...","[remember, tell, ask, forget, wish, inform, im...",[]
5,observing,"[monitoring, conducting, analyzing, photograph...","[monitoring, conducting, analyzing, photograph...",[]
6,told,"[spoke, talked, insisted, informed, explained,...","[spoke, talked, insisted, informed, explained,...",[]
7,formed,"[established, joined, created, incorporated, b...","[established, joined, created, incorporated, b...",[]
8,introduced,"[announced, created, modified, used, changed, ...","[announced, created, modified, used, changed, ...",[]
9,attracted,"[garnered, encouraged, generated, grew, grown,...","[garnered, encouraged, generated, grew, grown,...",[]


In [70]:
# =========================
# БЛОК 10.1. Диагностика после BERT-фильтра
# =========================

for _, row in df.head(8).iterrows():
    print("=" * 100)
    print("target:", row["target_clean"])
    print("target_score:", row["bert_scores"]["target_score"])
    print("top before BERT:", row["ranked_candidate_words_cefr"][:8])
    print("top after BERT:", row["ranked_candidate_words_bert"][:8])
    print("removed by BERT:", row["removed_by_bert_filter"][:5])

target: announced
target_score: 0.0006681207451038063
top before BERT: ['presented', 'agreed', 'planned', 'decided', 'extended', 'expected', 'unveiled', 'proposed']
top after BERT: ['presented', 'agreed', 'planned', 'decided', 'extended', 'expected', 'unveiled', 'proposed']
removed by BERT: []
target: interested
target_score: 2.008183037105482e-05
top before BERT: ['involved', 'realized']
top after BERT: ['involved', 'realized']
removed by BERT: []
target: however
target_score: 2.1621238133207044e-09
top before BERT: ['although', 'therefore', 'moreover', 'though', 'otherwise', 'instead']
top after BERT: ['although', 'therefore', 'moreover', 'though', 'otherwise', 'instead']
removed by BERT: []
target: thick
target_score: 9.259109035042457e-09
top before BERT: ['tall', 'dark', 'yellow', 'dry', 'shiny', 'rough', 'gray', 'large']
top after BERT: ['tall', 'dark', 'yellow', 'dry', 'shiny', 'rough', 'gray', 'large']
removed by BERT: []
target: remind
target_score: 1e-12
top before BERT: ['re

In [73]:
# =========================
# БЛОК 11. Финальный выбор 3 дистракторов
# =========================

FINAL_N = 3


def select_final_distractors_from_ranked(
    ranked_candidates: List[Dict],
    n: int = 3
) -> List[str]:
    """
    Берет первые n кандидатов из финального ranked списка.
    """
    selected = []

    for item in ranked_candidates:
        candidate = clean_text(item["candidate"]).lower()

        if not candidate:
            continue

        if candidate in selected:
            continue

        selected.append(candidate)

        if len(selected) >= n:
            break

    return selected


df["final_distractors"] = df["ranked_candidates_bert"].apply(
    lambda ranked: select_final_distractors_from_ranked(
        ranked,
        n=FINAL_N
    )
)

df["num_final_distractors"] = df["final_distractors"].apply(len)

df[
    [
        "kod",
        "task_id",
        "target_clean",
        "final_distractors",
        "num_final_distractors"
    ]
].head(50)

,kod,task_id,target_clean,final_distractors,num_final_distractors
0,09B13D,30,announced,"[presented, agreed, planned]",3
1,09B13D,31,interested,"[involved, realized]",2
2,09B13D,32,however,"[although, therefore, moreover]",3
3,09B13D,33,thick,"[tall, dark, yellow]",3
4,09B13D,34,remind,"[remember, tell, ask]",3
5,09B13D,35,observing,"[monitoring, conducting, analyzing]",3
6,09B13D,36,told,"[spoke, talked, insisted]",3
7,0EAA8F,32,formed,"[established, joined, created]",3
8,0EAA8F,33,introduced,"[announced, created, modified]",3
9,0EAA8F,34,attracted,"[garnered, encouraged, generated]",3


In [72]:
# =========================
# БЛОК 11. Диагностика финального выбора
# =========================

for _, row in df.head(20).iterrows():
    print("=" * 100)
    print("kod:", row["kod"], "| task_id:", row["task_id"])
    print("target:", row["target_clean"])
    print("gold:", row["gold_distractors"])
    print("final:", row["final_distractors"])
    print("num:", row["num_final_distractors"])

kod: 09B13D | task_id: 30
target: announced
gold: ['promoted', 'extended', 'presented']
final: ['presented', 'agreed', 'planned']
num: 3
kod: 09B13D | task_id: 31
target: interested
gold: ['delighted', 'inspired', 'excited']
final: ['involved', 'realized']
num: 2
kod: 09B13D | task_id: 32
target: however
gold: ['moreover', 'although', 'otherwise']
final: ['although', 'therefore', 'moreover']
num: 3
kod: 09B13D | task_id: 33
target: thick
gold: ['heavy', 'strong', 'rich']
final: ['tall', 'dark', 'yellow']
num: 3
kod: 09B13D | task_id: 34
target: remind
gold: ['review', 'remember', 'reflect']
final: ['remember', 'tell', 'ask']
num: 3
kod: 09B13D | task_id: 35
target: observing
gold: ['obtaining', 'offending', 'objecting']
final: ['monitoring', 'conducting', 'analyzing']
num: 3
kod: 09B13D | task_id: 36
target: told
gold: ['talked', 'spoke', 'said']
final: ['spoke', 'talked', 'insisted']
num: 3
kod: 0EAA8F | task_id: 32
target: formed
gold: ['kept', 'held', 'used']
final: ['established', 

In [74]:
# =========================
# БЛОК 12. Метрики совпадения с gold distractors
# =========================

def calculate_gold_overlap(final_distractors: List[str], gold_distractors: List[str]) -> Dict[str, float]:
    """
    Считает совпадение с реальными дистракторами ЕГЭ.

    exact_overlap_count: сколько финальных дистракторов совпало с gold.
    overlap_ratio: доля совпавших из 3.
    """
    final_set = set(
        clean_text(x).lower()
        for x in final_distractors
        if clean_text(x)
    )

    gold_set = set(
        clean_text(x).lower()
        for x in gold_distractors
        if clean_text(x)
    )

    overlap = final_set & gold_set

    return {
        "exact_overlap_count": len(overlap),
        "overlap_ratio": len(overlap) / max(len(gold_set), 1),
        "overlap_items": ", ".join(sorted(overlap))
    }


df["gold_overlap_info"] = df.apply(
    lambda row: calculate_gold_overlap(
        row["final_distractors"],
        row["gold_distractors"]
    ),
    axis=1
)

df["gold_overlap_count"] = df["gold_overlap_info"].apply(
    lambda x: x["exact_overlap_count"]
)

df["gold_overlap_ratio"] = df["gold_overlap_info"].apply(
    lambda x: x["overlap_ratio"]
)

df["gold_overlap_items"] = df["gold_overlap_info"].apply(
    lambda x: x["overlap_items"]
)

print("Среднее число совпадений с gold:", df["gold_overlap_count"].mean())
print("Средняя доля совпадения:", df["gold_overlap_ratio"].mean())

df["gold_overlap_count"].value_counts().sort_index()

Среднее число совпадений с gold: 0.5714285714285714
Средняя доля совпадения: 0.19047619047619047


,count
gold_overlap_count,
0,210
1,77
2,49
3,7


In [75]:
# =========================
# БЛОК 12. Подробная финальная таблица
# =========================

final_rows = []

for _, row in df.iterrows():
    result = {
        "item_key": row["item_key"],
        "kod": row["kod"],
        "title": row["title"],
        "task_id": row["task_id"],
        "target": row["target_clean"],
        "target_cefr": get_cefr_level(row["target_clean"]),
        "sentence_with_gap": row["sentence_with_gap_clean"],
        "sentence_with_answer": row["sentence_clean"],
        "gold_distractors": ", ".join(row["gold_distractors"]),
        "final_distractors": ", ".join(row["final_distractors"]),
        "num_final_distractors": row["num_final_distractors"],
        "gold_overlap_count": row["gold_overlap_count"],
        "gold_overlap_ratio": row["gold_overlap_ratio"],
        "gold_overlap_items": row["gold_overlap_items"]
    }

    for i in range(FINAL_N):
        if i < len(row["final_distractors"]):
            candidate = row["final_distractors"][i]

            # Ищем информацию о candidate в ranked_candidates_bert
            candidate_info = None

            for item in row["ranked_candidates_bert"]:
                if item["candidate"] == candidate:
                    candidate_info = item
                    break

            result[f"generated_distractor_{i+1}"] = candidate
            result[f"generated_distractor_{i+1}_cefr"] = get_cefr_level(candidate)

            if candidate_info is not None:
                result[f"generated_distractor_{i+1}_sources"] = ", ".join(candidate_info["sources"])
                result[f"generated_distractor_{i+1}_similarity"] = candidate_info["similarity"]
                result[f"generated_distractor_{i+1}_score"] = candidate_info["final_score"]
            else:
                result[f"generated_distractor_{i+1}_sources"] = ""
                result[f"generated_distractor_{i+1}_similarity"] = None
                result[f"generated_distractor_{i+1}_score"] = None
        else:
            result[f"generated_distractor_{i+1}"] = ""
            result[f"generated_distractor_{i+1}_cefr"] = ""
            result[f"generated_distractor_{i+1}_sources"] = ""
            result[f"generated_distractor_{i+1}_similarity"] = None
            result[f"generated_distractor_{i+1}_score"] = None

    final_rows.append(result)


final_results_df = pd.DataFrame(final_rows)

final_results_df.head()

,item_key,kod,title,task_id,target,target_cefr,sentence_with_gap,sentence_with_answer,gold_distractors,final_distractors,...,generated_distractor_2,generated_distractor_2_cefr,generated_distractor_2_sources,generated_distractor_2_similarity,generated_distractor_2_score,generated_distractor_3,generated_distractor_3_cefr,generated_distractor_3_sources,generated_distractor_3_similarity,generated_distractor_3_score
0,09B13D|Going on a hike|30,09B13D,Going on a hike,30,announced,B1,"Last week, our coach Mr Brown 30 ______ we wer...","Last week, our coach Mr Brown announced we wer...","promoted, extended, presented","presented, agreed, planned",...,agreed,A1,SemanticModel,0.770929,0.820929,planned,A1,SemanticModel,0.748850,0.798850
1,09B13D|Going on a hike|31,09B13D,Going on a hike,31,interested,A1,We weren't all that 31 ______ in the outdoors.,We weren't all that interested in the outdoors.,"delighted, inspired, excited","involved, realized",...,realized,None,SemanticModel,0.603800,0.653800,,,,NaN,NaN
2,09B13D|Going on a hike|32,09B13D,Going on a hike,32,however,A2,"32 ______, Mr Brown claimed that we'd really e...","However, Mr Brown claimed that we'd really enj...","moreover, although, otherwise","although, therefore, moreover",...,therefore,B1,"SemanticModel, LearnerCorpus",0.782361,1.082361,moreover,B1,"SemanticModel, LearnerCorpus",0.769414,1.069414
3,09B13D|Going on a hike|33,09B13D,Going on a hike,33,thick,A1,"Now we were in a state park, marching through ...","Now we were in a state park, marching through ...","heavy, strong, rich","tall, dark, yellow",...,dark,A1,SemanticModel,0.665432,0.715432,yellow,A1,SemanticModel,0.618783,0.668783
4,09B13D|Going on a hike|34,09B13D,Going on a hike,34,remind,A2,Doesn't its shape 34 ______ you of an elephant...,"Doesn't its shape remind you of an elephant?"" ...","review, remember, reflect","remember, tell, ask",...,tell,A1,SemanticModel,0.757472,0.807472,ask,A1,SemanticModel,0.677094,0.727094


In [76]:
# =========================
# БЛОК 12. Диагностическая таблица по всем кандидатам
# =========================

diagnostic_rows = []

for _, row in df.iterrows():
    final_set = set(row["final_distractors"])
    gold_set = set(row["gold_distractors"])

    for item in row["ranked_candidates_bert"]:
        candidate = item["candidate"]

        bert_info = row["bert_scores"]["candidate_scores"].get(
            candidate,
            {}
        )

        diagnostic_rows.append(
            {
                "item_key": row["item_key"],
                "kod": row["kod"],
                "title": row["title"],
                "task_id": row["task_id"],
                "target": row["target_clean"],
                "target_cefr": get_cefr_level(row["target_clean"]),
                "candidate": candidate,
                "candidate_cefr": get_cefr_level(candidate),
                "selected_final": candidate in final_set,
                "in_gold_distractors": candidate in gold_set,
                "sources": ", ".join(item["sources"]),
                "similarity": item["similarity"],
                "source_bonus": item["source_bonus"],
                "penalty": item["penalty"],
                "final_score": item["final_score"],
                "bert_candidate_score": bert_info.get("candidate_score"),
                "bert_target_score": bert_info.get("target_score"),
                "bert_relative_to_target": bert_info.get("relative_to_target")
            }
        )


diagnostic_df = pd.DataFrame(diagnostic_rows)

diagnostic_df.head()

,item_key,kod,title,task_id,target,target_cefr,candidate,candidate_cefr,selected_final,in_gold_distractors,sources,similarity,source_bonus,penalty,final_score,bert_candidate_score,bert_target_score,bert_relative_to_target
0,09B13D|Going on a hike|30,09B13D,Going on a hike,30,announced,B1,presented,A1,True,True,LearnerCorpus,0.610056,0.25,0.0,0.860056,4.550969e-11,0.000668,6.811597e-08
1,09B13D|Going on a hike|30,09B13D,Going on a hike,30,announced,B1,agreed,A1,True,False,SemanticModel,0.770929,0.05,0.0,0.820929,6.241898e-08,0.000668,9.342471e-05
2,09B13D|Going on a hike|30,09B13D,Going on a hike,30,announced,B1,planned,A1,True,False,SemanticModel,0.748850,0.05,0.0,0.798850,4.233681e-09,0.000668,6.336700e-06
3,09B13D|Going on a hike|30,09B13D,Going on a hike,30,announced,B1,decided,A2,False,False,SemanticModel,0.745480,0.05,0.0,0.795480,1.414245e-10,0.000668,2.116750e-07
4,09B13D|Going on a hike|30,09B13D,Going on a hike,30,announced,B1,extended,B1,False,True,LearnerCorpus,0.536892,0.25,0.0,0.786892,4.029006e-11,0.000668,6.030356e-08


In [77]:
# =========================
# БЛОК 12. Сводная статистика
# =========================

summary_overall = pd.DataFrame(
    [
        {
            "metric": "Количество заданий",
            "value": len(final_results_df)
        },
        {
            "metric": "Заданий с 3 дистракторами",
            "value": int((final_results_df["num_final_distractors"] == 3).sum())
        },
        {
            "metric": "Заданий с менее чем 3 дистракторами",
            "value": int((final_results_df["num_final_distractors"] < 3).sum())
        },
        {
            "metric": "Среднее число совпадений с gold",
            "value": float(final_results_df["gold_overlap_count"].mean())
        },
        {
            "metric": "Средняя доля совпадения с gold",
            "value": float(final_results_df["gold_overlap_ratio"].mean())
        }
    ]
)

source_rows = []

for i in range(1, FINAL_N + 1):
    source_col = f"generated_distractor_{i}_sources"

    for value in final_results_df[source_col].dropna():
        for source in str(value).split(","):
            source = source.strip()

            if source:
                source_rows.append(source)

summary_sources = (
    pd.Series(source_rows)
    .value_counts()
    .reset_index()
)

summary_sources.columns = [
    "source",
    "count"
]

cefr_rows = []

for i in range(1, FINAL_N + 1):
    cefr_col = f"generated_distractor_{i}_cefr"

    cefr_rows.extend(
        final_results_df[cefr_col]
        .replace("", pd.NA)
        .dropna()
        .tolist()
    )

summary_cefr = (
    pd.Series(cefr_rows)
    .value_counts()
    .reset_index()
)

summary_cefr.columns = [
    "cefr_level",
    "count"
]

print(summary_overall)
print(summary_sources)
print(summary_cefr)

                                metric       value
0                   Количество заданий  343.000000
1            Заданий с 3 дистракторами  325.000000
2  Заданий с менее чем 3 дистракторами   18.000000
3      Среднее число совпадений с gold    0.571429
4       Средняя доля совпадения с gold    0.190476
          source  count
0  SemanticModel    902
1  LearnerCorpus    291
  cefr_level  count
0         A1    443
1         A2    235
2         B1    208
3         B2     46


In [ ]:
                           metric       value
0                   Количество заданий  490.000000
1            Заданий с 3 дистракторами  472.000000
2  Заданий с менее чем 3 дистракторами   18.000000
3      Среднее число совпадений с gold    0.571429
4       Средняя доля совпадения с gold    0.190476
          source  count
0  SemanticModel    830
1  LearnerCorpus    363
  cefr_level  count
0         A1    443
1         A2    235
2         B1    208
3         B2     46

In [78]:
# =========================
# БЛОК 12. Экспорт результатов
# =========================

final_results_df.to_excel(
    "ege_generated_distractors_final.xlsx",
    index=False
)

diagnostic_df.to_excel(
    "ege_distractors_diagnostics_final.xlsx",
    index=False
)

with pd.ExcelWriter("ege_distractors_summary_final.xlsx") as writer:
    summary_overall.to_excel(
        writer,
        sheet_name="overall",
        index=False
    )

    summary_sources.to_excel(
        writer,
        sheet_name="sources",
        index=False
    )

    summary_cefr.to_excel(
        writer,
        sheet_name="cefr",
        index=False
    )

print("Файлы сохранены:")
print("ege_generated_distractors_final.xlsx")
print("ege_distractors_diagnostics_final.xlsx")
print("ege_distractors_summary_final.xlsx")

Файлы сохранены:
ege_generated_distractors_final.xlsx
ege_distractors_diagnostics_final.xlsx
ege_distractors_summary_final.xlsx


In [ ]:
# =========================
# БЛОК 12. Скачать файлы из Colab
# =========================

from google.colab import files

files.download("ege_generated_distractors_final.xlsx")
files.download("ege_distractors_diagnostics_final.xlsx")
files.download("ege_distractors_summary_final.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>